# Bibliotecas

In [1]:
# Bibliotecas gerais
import pandas as pd
import os
import re
import getpass
from datetime import date
import time
import base64
import regex 
from tqdm import tqdm
from langdetect import detect
import numpy as np
import unicodedata
import matplotlib.pyplot as plt
import json

# Biblioteca de requisições
import requests
from bs4 import BeautifulSoup
import urllib.parse
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver as uc

# Biblioteca para armazenar dados no banco de dados
from sqlalchemy import create_engine, Table, MetaData, insert, text
from sqlalchemy.dialects.mysql import insert
from sqlalchemy.exc import SQLAlchemyError

# Tradutor e bilbiotecas para numve de palavras
from deep_translator import GoogleTranslator
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from nltk.corpus import stopwords
import nltk
import spacy
from PIL import Image, ImageOps

# Configurações banco de dados

In [2]:
# Tenta recuperar a conexão com o banco, se definido antes
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "filmes_em_cartaz"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

    print("Configuração do Banco de Dados criada com Sucesso!!!")

else:
    print("Configuração já existe. Usando configuração já existente!!!")

Digite seu usuário MySQL:  tomida
Digite sua senha MySQL (não será exibida):  ········


Configuração do Banco de Dados criada com Sucesso!!!


# API KEY TMDB

In [6]:
# Tenta recuperar a conexão com o banco, se definido antes
api_key = globals().get("api_key", None)

if not api_key:
    api_key = getpass.getpass("Digite sua API KEY do TMDB: ")

    print("API KEY registrada com Sucesso!!!")

else:
    print("API KEY já existe. Usando API KEY já existente!!!")

Digite sua API KEY do TMDB:  ········


API KEY registrada com Sucesso!!!


# Coletar filmes em cartaz

In [11]:
def buscar_filmes_tmdb():
    """
    Descrição:
        Busca todos os filmes em cartaz na TMDb (Now Playing), baixando os posters
        e retornando um DataFrame com os títulos em português, inglês e o título original,
        além de data de estreia e caminhos dos posters.

    Processo:
        1. Recupera a chave de API global `api_key`.
        2. Cria a pasta `imagens/` para armazenar posters.
        3. Faz requisição inicial para determinar o total de páginas.
        4. Itera sobre todas as páginas de resultados:
            • Extrai título em português, título original e data de estreia.
            • Busca título em inglês usando o ID do filme.
            • Baixa e salva o poster localmente (se disponível).
            • Limpa caracteres inválidos nos nomes e nomes de arquivos.
        5. Agrega todas as informações em listas.
        6. Cria um DataFrame final com todas as colunas relevantes.

    Retorno:
        pandas.DataFrame:
            DataFrame contendo colunas:
                - nome_pt: Título em português
                - nome_original: Título original do filme
                - nome_en: Título em inglês
                - data_estreia: Data de lançamento
                - url_poster: URL do poster
                - caminho_imagem: Caminho local da imagem salva
                - sinopse: Sinopse do filme
                - nota_media: Nota média do filme no TMDB
                - num_avaliacoes: Número de avaliações do filme no TMDB
                - popularidade: Popularidade do filme no TMDB
    """

    # =============================================================
    # Preparação: chave de API e pasta de imagens
    # =============================================================
    api_key = globals().get("api_key", None)
    os.makedirs("imagens/pt", exist_ok=True)
    os.makedirs("imagens/en", exist_ok=True)

    url = "https://api.themoviedb.org/3/movie/now_playing"
    params = {
        "api_key": api_key,
        "language": "pt-BR",
        "region": "BR",
        "page": 1
    }

    # =============================================================
    # Listas para agregar informações
    # =============================================================
    lista_nome_pt = []
    lista_nome_original = []
    lista_nome_en = []
    lista_data_estreia = []
    lista_url_poster_pt = []
    lista_url_poster_en = []
    lista_caminho_imagem_pt = []
    lista_caminho_imagem_en = []
    lista_sinopse = []
    lista_nota_media = []
    lista_num_avaliacoes = []
    lista_popularidade = []

    # =============================================================
    # Requisição inicial para determinar total de páginas
    # =============================================================
    response = requests.get(url, params=params)
    data = response.json()
    total_pages = data.get("total_pages", 1)

    # =============================================================
    # Iteração sobre todas as páginas
    # =============================================================
    while params["page"] <= total_pages:
        print(f"[Página {params['page']} / {total_pages}]")

        response = requests.get(url, params=params)
        data = response.json()
        results = data.get("results", [])

        for movie in results:
            # ---------------------------------------------------------
            # Títulos e data de estreia
            # ---------------------------------------------------------
            nome_pt = re.sub(r'[\'"“”‘’]', '', movie.get("title", ""))
            nome_original = re.sub(r'[\'"“”‘’]', '', movie.get("original_title", ""))
            data_estreia = movie.get("release_date", "")
            url_poster_pt = movie.get("poster_path", "")
            sinopse = movie.get("overview", "")
            nota_media = movie.get("vote_average", "")
            num_avaliacoes = movie.get("vote_count", "")
            popularidade = movie.get("popularity", "")
            
            # ---------------------------------------------------------
            # Título em inglês usando ID do filme
            # ---------------------------------------------------------
            nome_en = ""
            movie_id = movie.get("id")
            if movie_id:
                url_en = f"https://api.themoviedb.org/3/movie/{movie_id}"
                params_en = {"api_key": api_key, "language": "en-US"}
                resp_en = requests.get(url_en, params=params_en).json()
                nome_en = re.sub(r'[\'"“”‘’]', '', resp_en.get("title", ""))
                url_poster_en = resp_en.get("poster_path", "")

            print(f"Processando PT-BR: {nome_pt:<10} | ORIGINAL: {nome_original:<10} | EN-US: {nome_en:<10}")

            # ---------------------------------------------------------
            # Poster e caminho local PT_BR
            # ---------------------------------------------------------
            caminho_relativo_pt = None
            if url_poster_pt:
                url_poster_pt_full = f"https://image.tmdb.org/t/p/original{url_poster_pt}"
                img = requests.get(url_poster_pt_full).content

                content_type = requests.head(url_poster_pt_full).headers.get("Content-Type", "")
                extensao = content_type.split("/")[-1] if "/" in content_type else "jpg"

                nome_arquivo = re.sub(r'[<>:"/\\|?*\'"“”‘’]', '', nome_pt)
                caminho_relativo_pt = f"imagens/pt/{nome_arquivo}.{extensao}"

                with open(caminho_relativo_pt, "wb") as f:
                    f.write(img)

                url_poster_pt = url_poster_pt_full

            # ---------------------------------------------------------
            # Poster e caminho local EN
            # ---------------------------------------------------------
            caminho_relativo_en = None
            if url_poster_en:
                url_poster_en_full = f"https://image.tmdb.org/t/p/original{url_poster_en}"
                img = requests.get(url_poster_en_full).content

                content_type = requests.head(url_poster_en_full).headers.get("Content-Type", "")
                extensao = content_type.split("/")[-1] if "/" in content_type else "jpg"

                nome_arquivo = re.sub(r'[<>:"/\\|?*\'"“”‘’]', '', nome_en)
                caminho_relativo_en = f"imagens/en/{nome_arquivo}.{extensao}"

                with open(caminho_relativo_en, "wb") as f:
                    f.write(img)

                url_poster_en = url_poster_en_full

            # ---------------------------------------------------------
            # Adiciona informações às listas
            # ---------------------------------------------------------
            lista_nome_pt.append(nome_pt)
            lista_nome_original.append(nome_original)
            lista_nome_en.append(nome_en)
            lista_data_estreia.append(data_estreia)
            lista_url_poster_pt.append(url_poster_pt)
            lista_url_poster_en.append(url_poster_en)
            lista_caminho_imagem_pt.append(caminho_relativo_pt)
            lista_caminho_imagem_en.append(caminho_relativo_en)
            lista_sinopse.append(sinopse)
            lista_nota_media.append(nota_media)
            lista_num_avaliacoes.append(num_avaliacoes)
            lista_popularidade.append(popularidade)

        params["page"] += 1

    # =============================================================
    # Cria DataFrame final
    # =============================================================
    df_final = pd.DataFrame({
        "nome_pt": lista_nome_pt,
        "nome_original": lista_nome_original,
        "nome_en": lista_nome_en,
        "data_estreia": lista_data_estreia,
        "url_poster_pt": lista_url_poster_pt,
        "url_poster_en": lista_url_poster_en,
        "caminho_imagem_pt": lista_caminho_imagem_pt,
        "caminho_imagem_en": lista_caminho_imagem_en,
        "sinopse": lista_sinopse,
        "nota_media": lista_nota_media,
        "num_avaliacoes": lista_num_avaliacoes,
        "popularidade": lista_popularidade
    })

    return df_final

In [12]:
%%time
df_filmes = buscar_filmes_tmdb()

[Página 1 / 8]
Processando PT-BR: Missão Refúgio | ORIGINAL: Shelter    | EN-US: Shelter   
Processando PT-BR: Cara de Um, Focinho de Outro | ORIGINAL: Hoppers    | EN-US: Hoppers   
Processando PT-BR: Destruição Final 2 | ORIGINAL: Greenland 2: Migration | EN-US: Greenland 2: Migration
Processando PT-BR: O Som da Morte | ORIGINAL: Whistle    | EN-US: Whistle   
Processando PT-BR: O Morro dos Ventos Uivantes | ORIGINAL: Wuthering Heights | EN-US: Wuthering Heights
Processando PT-BR: Pânico 7   | ORIGINAL: Scream 7   | EN-US: Scream 7  
Processando PT-BR: Caminhos do Crime | ORIGINAL: Crime 101  | EN-US: Crime 101 
Processando PT-BR: Um Cabra Bom de Bola | ORIGINAL: GOAT       | EN-US: GOAT      
Processando PT-BR: A Noiva!   | ORIGINAL: The Bride! | EN-US: The Bride!
Processando PT-BR: O Testamento de Ann Lee | ORIGINAL: The Testament of Ann Lee | EN-US: The Testament of Ann Lee
Processando PT-BR: A Viagem de Chihiro | ORIGINAL: 千と千尋の神隠し   | EN-US: Spirited Away
Processando PT-BR: Sonh

In [14]:
df_filmes.shape

(148, 12)

In [16]:
df_filmes.sort_values(by = 'popularidade', ascending = False).head(20)

,nome_pt,nome_original,nome_en,data_estreia,url_poster_pt,url_poster_en,caminho_imagem_pt,caminho_imagem_en,sinopse,nota_media,num_avaliacoes,popularidade
0,Missão Refúgio,Shelter,Shelter,2026-03-12,https://image.tmdb.org/t/p/original/hSvhZRkbYD...,https://image.tmdb.org/t/p/original/buPFnHZ3xQ...,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg,"Um ex-assassino de aluguel, agora um homem rec...",6.596,307,298.5287
1,"Cara de Um, Focinho de Outro",Hoppers,Hoppers,2026-03-05,https://image.tmdb.org/t/p/original/ftPFJBGbld...,https://image.tmdb.org/t/p/original/xjtWQ2CL1m...,"imagens/pt/Cara de Um, Focinho de Outro.jpeg",imagens/en/Hoppers.jpeg,Mabel é uma jovem amante da natureza que tenta...,7.718,195,148.4862
2,Destruição Final 2,Greenland 2: Migration,Greenland 2: Migration,2026-02-05,https://image.tmdb.org/t/p/original/sgvNCaWe4L...,https://image.tmdb.org/t/p/original/z2tqCJLsw6...,imagens/pt/Destruição Final 2.jpeg,imagens/en/Greenland 2 Migration.jpeg,Tendo encontrado a segurança do bunker da Groe...,6.437,608,138.5129
3,O Som da Morte,Whistle,Whistle,2026-02-05,https://image.tmdb.org/t/p/original/1tX11KQzuj...,https://image.tmdb.org/t/p/original/84vAcFwxoO...,imagens/pt/O Som da Morte.jpeg,imagens/en/Whistle.jpeg,Um grupo de estudantes disfuncionais se depara...,6.106,151,108.8578
4,O Morro dos Ventos Uivantes,Wuthering Heights,Wuthering Heights,2026-02-12,https://image.tmdb.org/t/p/original/762vOiAWzo...,https://image.tmdb.org/t/p/original/3YBce6dTh1...,imagens/pt/O Morro dos Ventos Uivantes.jpeg,imagens/en/Wuthering Heights.jpeg,O órfão Heathcliff é acolhido ainda criança pe...,6.355,516,106.4394
6,Caminhos do Crime,Crime 101,Crime 101,2026-02-12,https://image.tmdb.org/t/p/original/hHnEDdzdmK...,https://image.tmdb.org/t/p/original/heMdO64ys1...,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg,Um ladrão enigmático realiza assaltos de alto ...,7.100,174,104.6717
5,Pânico 7,Scream 7,Scream 7,2026-02-26,https://image.tmdb.org/t/p/original/oz4Mt34HH8...,https://image.tmdb.org/t/p/original/jjyuk0edLi...,imagens/pt/Pânico 7.jpeg,imagens/en/Scream 7.jpeg,Quando um novo Ghostface surge na pacata cidad...,6.000,320,77.3621
7,Um Cabra Bom de Bola,GOAT,GOAT,2026-02-12,https://image.tmdb.org/t/p/original/x0SRTrltSJ...,https://image.tmdb.org/t/p/original/wfuqMlaExc...,imagens/pt/Um Cabra Bom de Bola.jpeg,imagens/en/GOAT.jpeg,Uma pequena cabra com grandes sonhos recebe um...,7.600,110,50.0060
8,A Noiva!,The Bride!,The Bride!,2026-03-05,https://image.tmdb.org/t/p/original/vcdnSEcm0f...,https://image.tmdb.org/t/p/original/lV8YHwGkYZ...,imagens/pt/A Noiva!.jpeg,imagens/en/The Bride!.jpeg,"Na década de 1930, Frankenstein viaja a Chicag...",6.274,166,43.5131
9,O Testamento de Ann Lee,The Testament of Ann Lee,The Testament of Ann Lee,2026-03-12,https://image.tmdb.org/t/p/original/jeZBNXstpD...,https://image.tmdb.org/t/p/original/nb4LpzoNCN...,imagens/pt/O Testamento de Ann Lee.jpeg,imagens/en/The Testament of Ann Lee.jpeg,"Ann Lee, a líder fundadora do movimento Shaker...",6.703,64,31.1052


# Inserir no Banco de Dados e Salvar Parquet

In [17]:
# Salvar Parquet
os.makedirs("dados/filmes_em_cartaz", exist_ok=True)
df_filmes.to_parquet(f"dados/filmes_em_cartaz/filmes_em_cartaz_{date.today().strftime('%Y_%m_%d')}.parquet", index=False)

# Inserir no banco de dados
df_filmes.to_sql(
    'filmes_em_cartaz',
    con=db_engine,
    if_exists='append',
    index=False,
    method="multi",
    chunksize=5000
)

148

Seguiremos apenas com o top 10 de popularidade no TMDB:

In [18]:
df_filmes_top_10 = df_filmes.sort_values(by = 'popularidade', ascending = False).head(10).copy()

In [19]:
df_filmes_top_10

,nome_pt,nome_original,nome_en,data_estreia,url_poster_pt,url_poster_en,caminho_imagem_pt,caminho_imagem_en,sinopse,nota_media,num_avaliacoes,popularidade
0,Missão Refúgio,Shelter,Shelter,2026-03-12,https://image.tmdb.org/t/p/original/hSvhZRkbYD...,https://image.tmdb.org/t/p/original/buPFnHZ3xQ...,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg,"Um ex-assassino de aluguel, agora um homem rec...",6.596,307,298.5287
1,"Cara de Um, Focinho de Outro",Hoppers,Hoppers,2026-03-05,https://image.tmdb.org/t/p/original/ftPFJBGbld...,https://image.tmdb.org/t/p/original/xjtWQ2CL1m...,"imagens/pt/Cara de Um, Focinho de Outro.jpeg",imagens/en/Hoppers.jpeg,Mabel é uma jovem amante da natureza que tenta...,7.718,195,148.4862
2,Destruição Final 2,Greenland 2: Migration,Greenland 2: Migration,2026-02-05,https://image.tmdb.org/t/p/original/sgvNCaWe4L...,https://image.tmdb.org/t/p/original/z2tqCJLsw6...,imagens/pt/Destruição Final 2.jpeg,imagens/en/Greenland 2 Migration.jpeg,Tendo encontrado a segurança do bunker da Groe...,6.437,608,138.5129
3,O Som da Morte,Whistle,Whistle,2026-02-05,https://image.tmdb.org/t/p/original/1tX11KQzuj...,https://image.tmdb.org/t/p/original/84vAcFwxoO...,imagens/pt/O Som da Morte.jpeg,imagens/en/Whistle.jpeg,Um grupo de estudantes disfuncionais se depara...,6.106,151,108.8578
4,O Morro dos Ventos Uivantes,Wuthering Heights,Wuthering Heights,2026-02-12,https://image.tmdb.org/t/p/original/762vOiAWzo...,https://image.tmdb.org/t/p/original/3YBce6dTh1...,imagens/pt/O Morro dos Ventos Uivantes.jpeg,imagens/en/Wuthering Heights.jpeg,O órfão Heathcliff é acolhido ainda criança pe...,6.355,516,106.4394
6,Caminhos do Crime,Crime 101,Crime 101,2026-02-12,https://image.tmdb.org/t/p/original/hHnEDdzdmK...,https://image.tmdb.org/t/p/original/heMdO64ys1...,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg,Um ladrão enigmático realiza assaltos de alto ...,7.100,174,104.6717
5,Pânico 7,Scream 7,Scream 7,2026-02-26,https://image.tmdb.org/t/p/original/oz4Mt34HH8...,https://image.tmdb.org/t/p/original/jjyuk0edLi...,imagens/pt/Pânico 7.jpeg,imagens/en/Scream 7.jpeg,Quando um novo Ghostface surge na pacata cidad...,6.000,320,77.3621
7,Um Cabra Bom de Bola,GOAT,GOAT,2026-02-12,https://image.tmdb.org/t/p/original/x0SRTrltSJ...,https://image.tmdb.org/t/p/original/wfuqMlaExc...,imagens/pt/Um Cabra Bom de Bola.jpeg,imagens/en/GOAT.jpeg,Uma pequena cabra com grandes sonhos recebe um...,7.600,110,50.0060
8,A Noiva!,The Bride!,The Bride!,2026-03-05,https://image.tmdb.org/t/p/original/vcdnSEcm0f...,https://image.tmdb.org/t/p/original/lV8YHwGkYZ...,imagens/pt/A Noiva!.jpeg,imagens/en/The Bride!.jpeg,"Na década de 1930, Frankenstein viaja a Chicag...",6.274,166,43.5131
9,O Testamento de Ann Lee,The Testament of Ann Lee,The Testament of Ann Lee,2026-03-12,https://image.tmdb.org/t/p/original/jeZBNXstpD...,https://image.tmdb.org/t/p/original/nb4LpzoNCN...,imagens/pt/O Testamento de Ann Lee.jpeg,imagens/en/The Testament of Ann Lee.jpeg,"Ann Lee, a líder fundadora do movimento Shaker...",6.703,64,31.1052


In [60]:
# Inserir na tabela notas_filmes_em_cartaz
# Salvar Parquet
os.makedirs("dados/tmdb/notas", exist_ok=True)

df_filmes_top_10_tmdb = df_filmes_top_10[['nome_pt', 'nome_original', 'nome_en', 'nota_media']].copy()
df_filmes_top_10_tmdb = df_filmes_top_10_tmdb.rename(columns={"nota_media": "nota"})
df_filmes_top_10_tmdb['fonte'] = 'TMDB'

# Percorre cada linha do DataFrame
for idx, row in df_filmes_top_10_tmdb.iterrows():
    nome_pt = row['nome_pt']  
    nome_original = row['nome_original']  
    nome_en = row['nome_en']  
    nota = row['nota']
    fonte = row['fonte']

    # Remove caracteres inválidos para o nome do arquivo
    nome_arquivo = re.sub(r'[<>:"/\\|?*\'"“”‘’]', '', nome_pt)

    # Salva Parquet individual
    caminho_parquet = f"dados/tmdb/notas/{nome_arquivo}.parquet"
    row_df = pd.DataFrame([row])  # transforma a linha em DataFrame
    row_df.to_parquet(caminho_parquet, index=False)

    # Inserir no banco
    row_df.to_sql(
        'notas_filmes_em_cartaz',
        con=db_engine,
        if_exists='append',
        index=False,
        method="multi",
        chunksize=5000
    )

    print(f"Salvo e inserido: {nome_pt}")

Salvo e inserido: Missão Refúgio
Salvo e inserido: O Som da Morte
Salvo e inserido: Cara de Um, Focinho de Outro
Salvo e inserido: Destruição Final 2
Salvo e inserido: Pânico 7
Salvo e inserido: O Morro dos Ventos Uivantes
Salvo e inserido: Caminhos do Crime
Salvo e inserido: A Noiva!
Salvo e inserido: Um Cabra Bom de Bola
Salvo e inserido: O Testamento de Ann Lee


# Coletar Notas e Comentários dos filmes em cartaz

## IMDB

In [4]:
def buscar_reviews_imdb(nome_filme, max_reviews=50):
    """
    Descrição:
        Busca reviews de usuários de um filme no IMDb, incluindo a nota geral do filme.
        Retorna até `max_reviews` reviews.
        Apenas reviews com nota e texto válidos são incluídos, e emojis são removidos.

    Parâmetros:
        nome_filme (str): Nome do filme a ser pesquisado no IMDb.
        max_reviews (int): Número máximo de reviews a serem retornadas.

    Processo:
        1. Abre o IMDb usando Selenium em modo headless.
        2. Pesquisa o filme pelo nome e seleciona o primeiro resultado.
        3. Extrai a nota geral do filme exibida na página.
        4. Navega até a página de reviews do filme.
        5. Coleta os reviews e as notas dos usuários:
            - Remove emojis do texto.
            - Descarta reviews sem nota ou texto válido.
        6. Percorre as páginas de reviews até atingir `max_reviews`.

    Retorno:
        tuple:
            - pandas.DataFrame: DataFrame com colunas ["nota", "review"], notas na escala 1–10.
            - float | None: Nota geral do filme (escala 1–10).
    """

    # =============================================================
    # Função auxiliar para remover emojis
    # =============================================================
    def remove_emojis(text):
        # Emojis principais
        text = regex.sub(r'\p{Extended_Pictographic}', '', text)
        
        # Skin tones
        text = regex.sub(r'[\U0001F3FB-\U0001F3FF]', '', text)
        
        # Bandeiras
        text = regex.sub(r'[\U0001F1E6-\U0001F1FF]{2}', '', text)
        
        # Keycaps 
        text = regex.sub(r'[0-9#*]\uFE0F?\u20E3', '', text)
        
        # Caracteres invisíveis
        text = regex.sub(r'[\u200D\uFE0E\uFE0F]', '', text)
        
        # Limpeza final
        text = regex.sub(r'\s+', ' ', text).strip()
        
        return text
        
    # =============================================================
    # Configuração do navegador Chrome headless
    # =============================================================
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)

    try:
        # =============================================================
        # 1️ Buscar IMDb ID do filme
        # =============================================================
        query = urllib.parse.quote(nome_filme)
        url = f"https://www.imdb.com/find?q={query}&s=tt"
        driver.get(url)

        wait = WebDriverWait(driver, 30)
        wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'section[data-testid="find-results-section-title"]')
            )
        )

        # Tenta pegar o primeiro resultado
        links = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/title/tt"]')
        if not links:
            print(f"Nenhum resultado encontrado para '{nome_filme}'")
            return pd.DataFrame(columns=["nota", "review"]), None
        
        href = links[0].get_attribute("href")
        imdb_id = href.split("/title/")[1].split("/")[0]

        # =============================================================
        # 2️ Obter nota geral do filme
        # =============================================================
        driver.get(f"https://www.imdb.com/title/{imdb_id}/")
        time.sleep(2)

        try:
            nota_geral_el = driver.find_element(
                By.CSS_SELECTOR, '[data-testid="hero-rating-bar__aggregate-rating__score"] span'
            )
            nota_geral = float(nota_geral_el.text.replace(",", "."))
        except:
            nota_geral = None

        # =============================================================
        # 3️ Obter reviews de usuários
        # =============================================================
        driver.get(f"https://www.imdb.com/title/{imdb_id}/reviews")
        time.sleep(2)

        # -------------------------------------------------------------
        # Clicar no botão "Ver tudo" para expandir reviews, se existir
        # -------------------------------------------------------------
        try:
            ver_tudo_btn = driver.find_element(
                By.CSS_SELECTOR, 
                "button.ipc-see-more__button"
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", ver_tudo_btn)
            time.sleep(1)
            ver_tudo_btn.click()
            time.sleep(2)  # espera a expansão carregar
        except:
            pass  # se não existir, continua normalmente

        notas = []
        reviews = []

        while len(reviews) < max_reviews:
            # ---------------------------------------------------------
            # Coleta os cards de review visíveis
            # ---------------------------------------------------------
            cards = driver.find_elements(By.CSS_SELECTOR, ".ipc-list-card__content")
            for card in cards:
                # ================== Nota ==================
                try:
                    nota_usuario_el = card.find_element(By.CSS_SELECTOR, ".ipc-rating-star--rating")
                    nota_usuario = nota_usuario_el.text.strip()
                    if not nota_usuario:
                        nota_usuario = None
                except:
                    nota_usuario = None
            
                # ================== Review ==================
                try:
                    review_el = card.find_element(By.CSS_SELECTOR, "h3.ipc-title__text")
                    review_text = review_el.text.strip()
                    review_text = remove_emojis(review_text)  # remove emojis
                    if not review_text:  # se vazio depois de remover emojis
                        review_text = None
                except:
                    review_text = None
            
                # ================== Salvar ==================
                if review_text is not None and nota_usuario is not None:
                    reviews.append(review_text)
                    notas.append(nota_usuario)

                if len(reviews) >= max_reviews:
                    break

            # ---------------------------------------------------------
            # Tenta clicar em "Ver mais reviews" se houver
            # ---------------------------------------------------------
            try:
                load_more = driver.find_element(By.CSS_SELECTOR, ".load-more-data")
                driver.execute_script("arguments[0].click();", load_more)
                time.sleep(2)
            except:
                break  # não há mais reviews para carregar

        # =============================================================
        # 4️ Cria DataFrame final e retorna
        # =============================================================
        df_reviews = pd.DataFrame({"nota": notas, "review": reviews})

        # Substitui vírgula por ponto e converte para float
        if not df_reviews.empty:
            df_reviews['nota'] = df_reviews['nota'].str.replace(",", ".", regex=False).astype(float)

        return df_reviews, nota_geral

    finally:
        # =============================================================
        # Encerra o driver do Selenium
        # =============================================================
        driver.quit()

In [5]:
%%time
df_imdb, nota_imdb = buscar_reviews_imdb("Scream 7")

print("Nota IMDb:", nota_imdb)
df_imdb.head()

Nota IMDb: 5.8
CPU times: total: 328 ms
Wall time: 21.4 s


,nota,review
0,6.0,the The ending lands with a dull echo rather t...
1,4.0,The killer reveal ruined this movie
2,2.0,Scream 3 looks amazing now
3,4.0,Some Cool Ideas in a Very Disappointing Entry
4,6.0,"Even with a good pace, it's clear that the fra..."


In [6]:
df_imdb.shape

(49, 2)

In [7]:
def salvar_reviews_e_nota_imdb(df_filmes):
    """
    Processa filmes do DataFrame fornecido, buscando notas e reviews no IMDb.
    
    Para cada filme na coluna 'nome' do DataFrame:
    1. Verifica se já existe uma nota do IMDb na tabela 'notas_filmes_em_cartaz'.
    2. Caso não exista, busca reviews e nota geral usando `buscar_reviews_imdb`.
    3. Salva os reviews na tabela 'reviews_filmes_em_cartaz' com a fonte 'IMDB'.
    4. Salva a nota geral na tabela 'notas_filmes_em_cartaz' com a fonte 'IMDB'.
    5. Gera arquivos parquet separados em 'dados/imdb/' para nota e reviews.
    
    Parâmetros:
        df_filmes (pd.DataFrame): DataFrame contendo pelo menos a coluna 'nome_pt'.
        db_engine (sqlalchemy.engine.Engine): Conexão com o banco de dados.
    
    Retorno:
        None
    """

    # Verificando conexão com o Banco de Dados    
    db_engine = globals().get("db_engine", None)

    if not db_engine:
        # Host e Database
        host = "localhost"
        database = "filmes_em_cartaz"
        
        # Solicita usuário e senha de forma segura
        user = input("Digite seu usuário MySQL: ")
        password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
        
        # Cria a engine SQLAlchemy
        db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

    # Criando diretórios, se não existir
    os.makedirs("dados/imdb/notas", exist_ok=True)
    os.makedirs("dados/imdb/reviews", exist_ok=True)

    # -------------------------------------------------------------
    # 1. Busca todos os filmes já processados
    # -------------------------------------------------------------
    query = "SELECT nome_pt FROM notas_filmes_em_cartaz WHERE fonte = 'IMDB'"
    with db_engine.connect() as conn:
        nomes_processados = pd.read_sql(query, conn)['nome_pt'].tolist()

    # Filtra apenas os filmes que ainda não foram processados
    df_filmes_a_processar = df_filmes[~df_filmes['nome_pt'].isin(nomes_processados)].copy()
    total = len(df_filmes_a_processar)

    # Filmes não processados
    lista_erros = []

    if total == 0:
        print("Todos os filmes já foram processados!")
        return

    # -------------------------------------------------------------
    # 2. Loop sobre os filmes restantes
    # -------------------------------------------------------------
    for idx, row in enumerate(df_filmes_a_processar.itertuples(), 1):
        nome_filme_en = row.nome_en
        nome_filme_pt = row.nome_pt
        nome_filme_original = row.nome_original

        # Print de progresso
        print("-" * 60)
        print(f"[{idx}/{total}] Processando: '{nome_filme_en}'")

        # Normaliza o nome para arquivos
        nome_normalizado = re.sub(r"[^\w\d]+", "_", nome_filme_pt)

        # Busca reviews e nota
        df_reviews, nota_geral = buscar_reviews_imdb(nome_filme_en)

        if df_reviews.empty:
            print(f"Nenhum review encontrado para '{nome_filme_en}'")
            lista_erros.append(nome_filme_en)
            continue

        # Adiciona coluna fonte e nome
        df_reviews['nome_en'] = nome_filme_en
        df_reviews['nome_pt'] = nome_filme_pt
        df_reviews['nome_original'] = nome_filme_original
        df_reviews['fonte'] = 'IMDB'

        # -------------------------------------------------------------
        # Salva reviews no SQL
        # -------------------------------------------------------------
        df_reviews_sql = df_reviews[['nome_pt', 'nome_original', 'nome_en', 'nota', 'review', 'fonte']]
        df_reviews_sql.to_sql(
            'reviews_filmes_em_cartaz',
            con=db_engine,
            if_exists='append',
            index=False,
            method="multi",
            chunksize=5000
        )

        # -------------------------------------------------------------
        # Salva nota no SQL e parquet
        # -------------------------------------------------------------
        if nota_geral is not None:
            df_nota_sql = pd.DataFrame({
                'nome_pt': [nome_filme_pt],
                'nome_original': [nome_filme_original],
                'nome_en': [nome_filme_en],
                'nota': [nota_geral],
                'fonte': ['IMDB']
            })

            df_nota_sql.to_sql(
                'notas_filmes_em_cartaz',
                con=db_engine,
                if_exists='append',
                index=False,
                method="multi",
                chunksize=5000
            )

            # Salvando arquivos .parquet
            df_nota_sql.to_parquet(f"dados/imdb/notas/{nome_normalizado}.parquet", index=False)
            df_reviews_sql.to_parquet(f"dados/imdb/reviews/{nome_normalizado}.parquet", index=False)

        print(f"Concluído: '{nome_filme_en}'\n")

        # Pausa entre requisições
        time.sleep(10)

    print("-" * 60)
    print("-" * 60)
    print("Processo Finalizado com sucesso!!!")
    print(f"{len(lista_erros)} Filmes sem Reviews")
    print("-" * 60)
    print("-" * 60)

In [8]:
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "filmes_em_cartaz"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

# Lista de filmes
query = "SELECT * FROM filmes_em_cartaz"
df_filmes = pd.read_sql(query, con=db_engine)
df_filmes_top_10 = df_filmes.sort_values(by = 'popularidade', ascending = False).head(10).copy()

In [9]:
df_filmes_top_10.shape

(10, 12)

In [10]:
df_filmes_top_10

,id,nome_pt,nome_original,nome_en,data_estreia,url_poster,caminho_imagem,sinopse,nota_media,num_avaliacoes,popularidade,created_at
0,1,Missão Refúgio,Shelter,Shelter,2026-03-12,https://image.tmdb.org/t/p/original/hSvhZRkbYD...,imagens/Missão Refúgio.jpeg,"Um ex-assassino de aluguel, agora um homem rec...",6.65,284,294.6049,2026-03-14 09:36:01
1,2,O Som da Morte,Whistle,Whistle,2026-02-05,https://image.tmdb.org/t/p/original/1tX11KQzuj...,imagens/O Som da Morte.jpeg,Um grupo de estudantes disfuncionais se depara...,6.13,131,141.9966,2026-03-14 09:36:01
2,3,"Cara de Um, Focinho de Outro",Hoppers,Hoppers,2026-03-05,https://image.tmdb.org/t/p/original/ftPFJBGbld...,"imagens/Cara de Um, Focinho de Outro.jpeg",Mabel é uma jovem amante da natureza que tenta...,7.70,158,139.8369,2026-03-14 09:36:01
3,4,Destruição Final 2,Greenland 2: Migration,Greenland 2: Migration,2026-02-05,https://image.tmdb.org/t/p/original/sgvNCaWe4L...,imagens/Destruição Final 2.jpeg,Tendo encontrado a segurança do bunker da Groe...,6.46,587,131.5286,2026-03-14 09:36:01
4,5,Pânico 7,Scream 7,Scream 7,2026-02-26,https://image.tmdb.org/t/p/original/oz4Mt34HH8...,imagens/Pânico 7.jpeg,Quando um novo Ghostface surge na pacata cidad...,5.95,297,91.6557,2026-03-14 09:36:01
5,6,O Morro dos Ventos Uivantes,Wuthering Heights,Wuthering Heights,2026-02-12,https://image.tmdb.org/t/p/original/762vOiAWzo...,imagens/O Morro dos Ventos Uivantes.jpeg,O órfão Heathcliff é acolhido ainda criança pe...,6.40,490,75.0383,2026-03-14 09:36:01
6,7,Caminhos do Crime,Crime 101,Crime 101,2026-02-12,https://image.tmdb.org/t/p/original/hHnEDdzdmK...,imagens/Caminhos do Crime.jpeg,Um ladrão enigmático realiza assaltos de alto ...,7.02,155,64.5261,2026-03-14 09:36:01
7,8,A Noiva!,The Bride!,The Bride!,2026-03-05,https://image.tmdb.org/t/p/original/vcdnSEcm0f...,imagens/A Noiva!.jpeg,"Na década de 1930, Frankenstein viaja a Chicag...",6.30,150,62.6113,2026-03-14 09:36:01
8,9,Um Cabra Bom de Bola,GOAT,GOAT,2026-02-12,https://image.tmdb.org/t/p/original/x0SRTrltSJ...,imagens/Um Cabra Bom de Bola.jpeg,Uma pequena cabra com grandes sonhos recebe um...,7.50,103,47.0737,2026-03-14 09:36:01
9,10,O Testamento de Ann Lee,The Testament of Ann Lee,The Testament of Ann Lee,2026-03-12,https://image.tmdb.org/t/p/original/mDiS3wF7sz...,imagens/O Testamento de Ann Lee.jpeg,"Ann Lee, a líder fundadora do movimento Shaker...",7.00,37,35.6559,2026-03-14 09:36:01


In [11]:
%%time
# Buscar e salvar notas e reviews
salvar_reviews_e_nota_imdb(df_filmes_top_10)

------------------------------------------------------------
[1/10] Processando: 'Shelter'
Concluído: 'Shelter'

------------------------------------------------------------
[2/10] Processando: 'Whistle'
Concluído: 'Whistle'

------------------------------------------------------------
[3/10] Processando: 'Hoppers'
Concluído: 'Hoppers'

------------------------------------------------------------
[4/10] Processando: 'Greenland 2: Migration'
Concluído: 'Greenland 2: Migration'

------------------------------------------------------------
[5/10] Processando: 'Scream 7'
Concluído: 'Scream 7'

------------------------------------------------------------
[6/10] Processando: 'Wuthering Heights'
Concluído: 'Wuthering Heights'

------------------------------------------------------------
[7/10] Processando: 'Crime 101'
Concluído: 'Crime 101'

------------------------------------------------------------
[8/10] Processando: 'The Bride!'
Concluído: 'The Bride!'

----------------------------------

## Rotten Tomatoes

In [12]:
def buscar_reviews_rotten_tomatoes(nome_filme, max_reviews=50):
    """
    Descrição:
        Busca reviews da audiência verificada de um filme no Rotten Tomatoes,
        incluindo a nota geral da audiência. Retorna até `max_reviews` reviews, 
        padronizando as notas na escala 1–10.
        Apenas reviews com nota e texto válidos são incluídos, e emojis são removidos.

    Parâmetros:
        nome_filme (str): Nome do filme a ser pesquisado no Rotten Tomatoes.
        max_reviews (int): Número máximo de reviews da audiência a serem retornadas.

    Processo:
        1. Abre o Rotten Tomatoes usando Selenium em modo headless.
        2. Pesquisa o filme pelo nome e seleciona o primeiro resultado.
        3. Extrai a nota geral da audiência exibida na página e converte para a escala 1–10.
        4. Obtém o identificador interno (emsID) do filme.
        5. Usa a API interna para coletar reviews da audiência verificada.
        6. Para cada review:
            - Remove emojis do texto.
            - Converte a nota da escala STAR (0–5) para 1–10.
            - Descartar reviews sem nota ou texto válido.
        7. Percorre as páginas de reviews até atingir `max_reviews`.

    Retorno:
        tuple:
            - pandas.DataFrame: DataFrame com colunas ["nota", "review"], notas na escala 1–10.
            - float | None: Nota geral da audiência (escala 1–10).
    """

    # =============================================================
    # Função auxiliar para remover emojis
    # =============================================================
    def remove_emojis(text):
        # Emojis principais
        text = regex.sub(r'\p{Extended_Pictographic}', '', text)
        
        # Skin tones
        text = regex.sub(r'[\U0001F3FB-\U0001F3FF]', '', text)
        
        # Bandeiras
        text = regex.sub(r'[\U0001F1E6-\U0001F1FF]{2}', '', text)
        
        # Keycaps 
        text = regex.sub(r'[0-9#*]\uFE0F?\u20E3', '', text)
        
        # Caracteres invisíveis
        text = regex.sub(r'[\u200D\uFE0E\uFE0F]', '', text)
        
        # Limpeza final
        text = regex.sub(r'\s+', ' ', text).strip()
        
        return text
    
    # =============================================================
    # Configuração do navegador headless
    # =============================================================
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=options)

    try:

        # =============================================================
        # 1 Buscar filme no Rotten Tomatoes
        # =============================================================
        query = urllib.parse.quote(nome_filme)
        url = f"https://www.rottentomatoes.com/search?search={query}"

        driver.get(url)

        wait = WebDriverWait(driver, 30)

        filtro_filmes = wait.until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'li.js-search-filter[data-filter="movie"]')
            )
        )
        filtro_filmes.click()

        primeiro_resultado = wait.until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'a[data-qa="info-name"]')
            )
        )
        primeiro_resultado.click()

        # =============================================================
        # 2 Obter nota da audiência
        # =============================================================
        wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'rt-text[slot="audience-score"]')
            )
        )

        try:
            nota = driver.find_element(
                By.CSS_SELECTOR,
                'rt-text[slot="audience-score"]'
            ).text

            nota_audiencia_geral = float(nota.replace("%", "")) / 10
        except:
            nota_audiencia_geral = None

        # =============================================================
        # 3 Buscar ID do filme (emsID)
        # =============================================================
        scripts = driver.find_elements(By.TAG_NAME, "script")

        movie_id = None

        for script in scripts:
            texto = script.get_attribute("innerHTML")
            match = re.search(r'"emsID"\s*:\s*"([^"]+)"', texto)
            if match:
                movie_id = match.group(1)
                break

        if not movie_id:
            return pd.DataFrame(columns=["nota", "review"]), nota_audiencia_geral

    finally:
        driver.quit()

    # =============================================================
    # 4 Configuração da API Rotten Tomatoes
    # =============================================================
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Referer": "https://www.rottentomatoes.com/"
    }

    reviews = []
    ratings = []
    after = ""

    # =============================================================
    # 5 Coletar reviews da audiência verificada
    # =============================================================
    while len(reviews) < max_reviews:

        url = f"https://www.rottentomatoes.com/napi/rtcf/v1/movies/{movie_id}/reviews"

        params = {
            "after": after,
            "pageCount": 50,
            "type": "audience",
            "verified": "true"
        }

        r = requests.get(url, headers=headers, params=params)

        if r.status_code != 200:
            break

        data = r.json()
        batch = data.get("reviews", [])

        if not batch:
            break

        for rev in batch:
            # ================== Review ==================
            review_text = rev.get("review")
            if review_text:
                review_text = review_text.strip()
                review_text = remove_emojis(review_text)  # remove emojis
                if not review_text:  # descarta vazios após remover emojis
                    review_text = None
            else:
                review_text = None
        
            # ================== Rating ==================
            rating = rev.get("rating")
            if rating and rating.startswith("STAR_"):
                try:
                    parts = rating.split("_")[1:]                      
                    rating = 2.0 * (float(parts[0]) + float(parts[1]) / 10) 
                except (ValueError, IndexError):
                    rating = None
            else:
                rating = None
        
            # ================== Salvar ==================
            if review_text is not None and rating is not None:
                reviews.append(review_text)
                ratings.append(rating)

            if len(reviews) >= max_reviews:
                break

        page_info = data.get("pageInfo", {})

        if not page_info.get("hasNextPage"):
            break

        after = page_info.get("endCursor")

    # =============================================================
    # 6 Criar DataFrame final
    # =============================================================
    df_reviews = pd.DataFrame({
        "nota": ratings,
        "review": reviews
    })

    return df_reviews, nota_audiencia_geral

In [13]:
%%time
df_rotten_tomatoes, nota_rotten_tomatoes = buscar_reviews_rotten_tomatoes('Scream 7')
print("Nota Rotten Tomatoes:", nota_rotten_tomatoes)
df_rotten_tomatoes.head()

Nota Rotten Tomatoes: 7.6
CPU times: total: 1.77 s
Wall time: 12.3 s


,nota,review
0,3.0,Story line was decent but the death scenes wer...
1,7.0,While I did think the reveal was a little unde...
2,9.0,I liked the graphic violence
3,1.0,Probably the worst movie in the franchise. not...
4,3.0,We miss like half an hour 45 minutes of the mo...


In [14]:
df_rotten_tomatoes.shape

(50, 2)

In [15]:
def salvar_reviews_e_nota_rotten_tomatoes(df_filmes):
    """
    Processa filmes do DataFrame fornecido, buscando notas e reviews no Rotten Tomatoes.
    
    Para cada filme na coluna 'nome' do DataFrame:
    1. Verifica se já existe uma nota do Rotten Tomatoes na tabela 'notas_filmes_em_cartaz'.
    2. Caso não exista, busca reviews e nota geral usando `buscar_reviews_rotten_tomatoes`.
    3. Salva os reviews na tabela 'reviews_filmes_em_cartaz' com a fonte 'ROTTEN_TOMATOES'.
    4. Salva a nota geral na tabela 'notas_filmes_em_cartaz' com a fonte 'ROTTEN_TOMATOES'.
    5. Gera arquivos parquet separados em 'dados/rotten_tomatoes/' para nota e reviews.
    
    Parâmetros:
        df_filmes (pd.DataFrame): DataFrame contendo pelo menos a coluna 'nome_pt'.
        db_engine (sqlalchemy.engine.Engine): Conexão com o banco de dados.
    
    Retorno:
        None
    """

    # Verificando conexão com o Banco de Dados    
    db_engine = globals().get("db_engine", None)

    if not db_engine:
        # Host e Database
        host = "localhost"
        database = "filmes_em_cartaz"
        
        # Solicita usuário e senha de forma segura
        user = input("Digite seu usuário MySQL: ")
        password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
        
        # Cria a engine SQLAlchemy
        db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

    # Criando diretórios, se não existir
    os.makedirs("dados/rotten_tomatoes/notas", exist_ok=True)
    os.makedirs("dados/rotten_tomatoes/reviews", exist_ok=True)

    # -------------------------------------------------------------
    # 1. Busca todos os filmes já processados
    # -------------------------------------------------------------
    query = "SELECT nome_pt FROM notas_filmes_em_cartaz WHERE fonte = 'ROTTEN_TOMATOES'"
    with db_engine.connect() as conn:
        nomes_processados = pd.read_sql(query, conn)['nome_pt'].tolist()

    # Filtra apenas os filmes que ainda não foram processados
    df_filmes_a_processar = df_filmes[~df_filmes['nome_pt'].isin(nomes_processados)].copy()
    total = len(df_filmes_a_processar)

    # Filmes não processados
    lista_erros = []

    if total == 0:
        print("Todos os filmes já foram processados!")
        return

    # -------------------------------------------------------------
    # 2. Loop sobre os filmes restantes
    # -------------------------------------------------------------
    for idx, row in enumerate(df_filmes_a_processar.itertuples(), 1):
        nome_filme_en = row.nome_en
        nome_filme_pt = row.nome_pt
        nome_filme_original = row.nome_original

        # Print de progresso
        print("-" * 60)
        print(f"[{idx}/{total}] Processando: '{nome_filme_en}'")

        # Normaliza o nome para arquivos
        nome_normalizado = re.sub(r"[^\w\d]+", "_", nome_filme_pt)

        # Busca reviews e nota
        df_reviews, nota_geral = buscar_reviews_rotten_tomatoes(nome_filme_en)

        if df_reviews.empty:
            print(f"Nenhum review encontrado para '{nome_filme_en}'")
            lista_erros.append(nome_filme_en)
            continue

        # Adiciona coluna fonte e nome
        df_reviews['nome_en'] = nome_filme_en
        df_reviews['nome_pt'] = nome_filme_pt
        df_reviews['nome_original'] = nome_filme_original
        df_reviews['fonte'] = 'ROTTEN_TOMATOES'

        # -------------------------------------------------------------
        # Salva reviews no SQL
        # -------------------------------------------------------------
        df_reviews_sql = df_reviews[['nome_pt', 'nome_original', 'nome_en', 'nota', 'review', 'fonte']]
        df_reviews_sql.to_sql(
            'reviews_filmes_em_cartaz',
            con=db_engine,
            if_exists='append',
            index=False,
            method="multi",
            chunksize=5000
        )

        # -------------------------------------------------------------
        # Salva nota no SQL e parquet
        # -------------------------------------------------------------
        if nota_geral is not None:
            df_nota_sql = pd.DataFrame({
                'nome_pt': [nome_filme_pt],
                'nome_original': [nome_filme_original],
                'nome_en': [nome_filme_en],
                'nota': [nota_geral],
                'fonte': ['ROTTEN_TOMATOES']
            })

            df_nota_sql.to_sql(
                'notas_filmes_em_cartaz',
                con=db_engine,
                if_exists='append',
                index=False,
                method="multi",
                chunksize=5000
            )

            # Salvando arquivos .parquet
            df_nota_sql.to_parquet(f"dados/rotten_tomatoes/notas/{nome_normalizado}.parquet", index=False)
            df_reviews_sql.to_parquet(f"dados/rotten_tomatoes/reviews/{nome_normalizado}.parquet", index=False)

        print(f"Concluído: '{nome_filme_en}'\n")

        # Pausa entre requisições
        time.sleep(10)

    print("-" * 60)
    print("-" * 60)
    print("Processo Finalizado com sucesso!!!")
    print(f"{len(lista_erros)} Filmes sem Reviews")
    print("-" * 60)
    print("-" * 60)

In [16]:
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "filmes_em_cartaz"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

# Lista de filmes
query = "SELECT * FROM filmes_em_cartaz"
df_filmes = pd.read_sql(query, con=db_engine)
df_filmes_top_10 = df_filmes.sort_values(by = 'popularidade', ascending = False).head(10).copy()

In [17]:
df_filmes_top_10.shape

(10, 12)

In [18]:
df_filmes_top_10

,id,nome_pt,nome_original,nome_en,data_estreia,url_poster,caminho_imagem,sinopse,nota_media,num_avaliacoes,popularidade,created_at
0,1,Missão Refúgio,Shelter,Shelter,2026-03-12,https://image.tmdb.org/t/p/original/hSvhZRkbYD...,imagens/Missão Refúgio.jpeg,"Um ex-assassino de aluguel, agora um homem rec...",6.65,284,294.6049,2026-03-14 09:36:01
1,2,O Som da Morte,Whistle,Whistle,2026-02-05,https://image.tmdb.org/t/p/original/1tX11KQzuj...,imagens/O Som da Morte.jpeg,Um grupo de estudantes disfuncionais se depara...,6.13,131,141.9966,2026-03-14 09:36:01
2,3,"Cara de Um, Focinho de Outro",Hoppers,Hoppers,2026-03-05,https://image.tmdb.org/t/p/original/ftPFJBGbld...,"imagens/Cara de Um, Focinho de Outro.jpeg",Mabel é uma jovem amante da natureza que tenta...,7.70,158,139.8369,2026-03-14 09:36:01
3,4,Destruição Final 2,Greenland 2: Migration,Greenland 2: Migration,2026-02-05,https://image.tmdb.org/t/p/original/sgvNCaWe4L...,imagens/Destruição Final 2.jpeg,Tendo encontrado a segurança do bunker da Groe...,6.46,587,131.5286,2026-03-14 09:36:01
4,5,Pânico 7,Scream 7,Scream 7,2026-02-26,https://image.tmdb.org/t/p/original/oz4Mt34HH8...,imagens/Pânico 7.jpeg,Quando um novo Ghostface surge na pacata cidad...,5.95,297,91.6557,2026-03-14 09:36:01
5,6,O Morro dos Ventos Uivantes,Wuthering Heights,Wuthering Heights,2026-02-12,https://image.tmdb.org/t/p/original/762vOiAWzo...,imagens/O Morro dos Ventos Uivantes.jpeg,O órfão Heathcliff é acolhido ainda criança pe...,6.40,490,75.0383,2026-03-14 09:36:01
6,7,Caminhos do Crime,Crime 101,Crime 101,2026-02-12,https://image.tmdb.org/t/p/original/hHnEDdzdmK...,imagens/Caminhos do Crime.jpeg,Um ladrão enigmático realiza assaltos de alto ...,7.02,155,64.5261,2026-03-14 09:36:01
7,8,A Noiva!,The Bride!,The Bride!,2026-03-05,https://image.tmdb.org/t/p/original/vcdnSEcm0f...,imagens/A Noiva!.jpeg,"Na década de 1930, Frankenstein viaja a Chicag...",6.30,150,62.6113,2026-03-14 09:36:01
8,9,Um Cabra Bom de Bola,GOAT,GOAT,2026-02-12,https://image.tmdb.org/t/p/original/x0SRTrltSJ...,imagens/Um Cabra Bom de Bola.jpeg,Uma pequena cabra com grandes sonhos recebe um...,7.50,103,47.0737,2026-03-14 09:36:01
9,10,O Testamento de Ann Lee,The Testament of Ann Lee,The Testament of Ann Lee,2026-03-12,https://image.tmdb.org/t/p/original/mDiS3wF7sz...,imagens/O Testamento de Ann Lee.jpeg,"Ann Lee, a líder fundadora do movimento Shaker...",7.00,37,35.6559,2026-03-14 09:36:01


In [19]:
%%time
# Buscar e salvar notas e reviews
salvar_reviews_e_nota_rotten_tomatoes(df_filmes_top_10)

------------------------------------------------------------
[1/10] Processando: 'Shelter'
Concluído: 'Shelter'

------------------------------------------------------------
[2/10] Processando: 'Whistle'
Concluído: 'Whistle'

------------------------------------------------------------
[3/10] Processando: 'Hoppers'
Concluído: 'Hoppers'

------------------------------------------------------------
[4/10] Processando: 'Greenland 2: Migration'
Concluído: 'Greenland 2: Migration'

------------------------------------------------------------
[5/10] Processando: 'Scream 7'
Concluído: 'Scream 7'

------------------------------------------------------------
[6/10] Processando: 'Wuthering Heights'
Concluído: 'Wuthering Heights'

------------------------------------------------------------
[7/10] Processando: 'Crime 101'
Concluído: 'Crime 101'

------------------------------------------------------------
[8/10] Processando: 'The Bride!'
Concluído: 'The Bride!'

----------------------------------

## Metacritic

In [20]:
def buscar_reviews_metacritic(nome_filme, max_reviews=50):
    """
    Descrição:
        Busca reviews da audiência de um filme no Metacritic, incluindo a nota geral da audiência.
        Retorna até `max_reviews` reviews, padronizando as notas na escala 1–10.
        Apenas reviews com nota e texto válidos são incluídos, e emojis são removidos.

    Parâmetros:
        nome_filme (str): Nome do filme a ser pesquisado no Metacritic.
        max_reviews (int): Número máximo de reviews da audiência a serem retornadas.

    Processo:
        1. Abre o Metacritic usando Selenium em modo headless.
        2. Pesquisa o filme pelo nome e seleciona o primeiro resultado.
        3. Navega até a página de user reviews ordenadas por "Recently Added".
        4. Extrai a nota geral da audiência exibida na página e converte para a escala 1–10.
        5. Coleta os reviews e as notas dos usuários:
            - Remove emojis do texto.
            - Converte a nota da escala 0–10 para 1–10.
            - Descartar reviews sem nota ou texto válido.
        6. Percorre as páginas de reviews até atingir `max_reviews`.

    Retorno:
        tuple:
            - pandas.DataFrame: DataFrame com colunas ["nota", "review"], notas na escala 1–10.
            - float | None: Nota geral da audiência (escala 1–10).
    """

    # =============================================================
    # Função auxiliar para remover emojis
    # =============================================================
    def remove_emojis(text):
        # Emojis principais
        text = regex.sub(r'\p{Extended_Pictographic}', '', text)
        
        # Skin tones
        text = regex.sub(r'[\U0001F3FB-\U0001F3FF]', '', text)
        
        # Bandeiras
        text = regex.sub(r'[\U0001F1E6-\U0001F1FF]{2}', '', text)
        
        # Keycaps 
        text = regex.sub(r'[0-9#*]\uFE0F?\u20E3', '', text)
        
        # Caracteres invisíveis
        text = regex.sub(r'[\u200D\uFE0E\uFE0F]', '', text)
        
        # Limpeza final
        text = regex.sub(r'\s+', ' ', text).strip()
        
        return text
    
    # =============================================================
    # Configuração do navegador headless
    # =============================================================
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=options)
    
    try:
        # =============================================================
        # 1. Buscar filme no Metacritic
        # =============================================================
        query = urllib.parse.quote(nome_filme)
        url_busca = f"https://www.metacritic.com/search/{query}"
        driver.get(url_busca)
        wait = WebDriverWait(driver, 30)
        
        # --- 1a. Fechar barra de cookies, se existir ---
        try:
            aceitar_cookies = wait.until(
                EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
            )
            aceitar_cookies.click()
            time.sleep(1)  # pequena pausa para a animação sumir
        except:
            pass  # se não houver, continua
        
        # --- 1b. Espera e seleciona o primeiro resultado correto ---
        primeiro_resultado = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'a.c-search-item.search-item__content')
            )
        )
        
        # --- 1c. Scroll até o elemento ---
        driver.execute_script("arguments[0].scrollIntoView(true);", primeiro_resultado)
        time.sleep(0.5)  # espera o scroll finalizar
        
        # --- 1d. Clicar de forma segura ---
        try:
            primeiro_resultado.click()
        except ElementClickInterceptedException:
            # fallback: clicar via JS se ainda estiver bloqueado
            driver.execute_script("arguments[0].click();", primeiro_resultado)
        
        # =============================================================
        # 2. Ir para a página de user reviews
        # =============================================================
        # esperar a URL mudar
        wait.until(
            lambda d: d.current_url != url_busca)
        url_reviews = driver.current_url + "user-reviews?sort-by=Recently+Added"
        driver.get(url_reviews)

        # =============================================================
        # 3. Obter nota geral da audiência
        # =============================================================
        try:
            nota_elem = wait.until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, 'div.c-siteReviewScore_background-user span')
                )
            )
            nota_audiencia_geral = float(nota_elem.text.strip())
        except:
            nota_audiencia_geral = None

        # =============================================================
        # 4. Coletar reviews individuais
        # =============================================================
        reviews = []
        ratings = []

        while len(reviews) < max_reviews:
            review_cards = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, 'div.review-card__content')
                )
            )

            for card in review_cards:
                if len(reviews) >= max_reviews:
                    break
            
                # ================== Nota ==================
                try:
                    nota = card.find_element(
                        By.CSS_SELECTOR, 'a > div > div.c-siteReviewScore span'
                    ).text.strip()
                    nota = float(nota) * 0.9 + 1
                except:
                    nota = None
            
                # ================== Review ==================
                try:
                    review_text = card.find_element(
                        By.CSS_SELECTOR, 'div.review-card__quote'
                    ).text.strip()
                    review_text = remove_emojis(review_text)  # remove emojis
                    if not review_text:  # descarta se vazio após remover emojis
                        review_text = None
                except:
                    review_text = None
            
                # ================== Guardar ==================
                if review_text is not None and nota is not None:
                    reviews.append(review_text)
                    ratings.append(nota)

                if len(reviews) >= max_reviews:
                    break

            # Paginação
            try:
                next_button = driver.find_element(By.CSS_SELECTOR, 'button.pagination_next')
                if 'disabled' in next_button.get_attribute('class'):
                    break
                else:
                    next_button.click()
                    wait.until(EC.staleness_of(review_cards[0]))
            except:
                break

    finally:
        driver.quit()

    # =============================================================
    # 5. Criar DataFrame final
    # =============================================================
    df_reviews = pd.DataFrame({
        "nota": ratings,
        "review": reviews
    })

    return df_reviews, nota_audiencia_geral

In [21]:
%%time
df_metacritic, nota_metacritic = buscar_reviews_metacritic('Scream 7')
print("Nota Metacritic:", nota_metacritic)
df_metacritic.head()

Nota Metacritic: 5.1
CPU times: total: 266 ms
Wall time: 12 s


,nota,review
0,1.9,“Scream 7” may be below your expectations! You...
1,4.6,Pior revelação de Ghostface da franquia. O fil...
2,1.0,A franchise that once redefined the genre has ...
3,7.3,"Actualmente me gustó, los personajes principal..."
4,6.4,"So this has had a lot of bad reviews, but a fe..."


In [22]:
df_metacritic.shape

(50, 2)

In [23]:
def salvar_reviews_e_nota_metacritic(df_filmes):
    """
    Processa filmes do DataFrame fornecido, buscando notas e reviews no Metacritic.
    
    Para cada filme na coluna 'nome' do DataFrame:
    1. Verifica se já existe uma nota do Metacritic na tabela 'notas_filmes_em_cartaz'.
    2. Caso não exista, busca reviews e nota geral usando `buscar_reviews_metacritic`.
    3. Salva os reviews na tabela 'reviews_filmes_em_cartaz' com a fonte 'METACRITIC'.
    4. Salva a nota geral na tabela 'notas_filmes_em_cartaz' com a fonte 'METACRITIC'.
    5. Gera arquivos parquet separados em 'dados/metacritic/' para nota e reviews.
    
    Parâmetros:
        df_filmes (pd.DataFrame): DataFrame contendo pelo menos a coluna 'nome_pt'.
        db_engine (sqlalchemy.engine.Engine): Conexão com o banco de dados.
    
    Retorno:
        None
    """

    # Verificando conexão com o Banco de Dados    
    db_engine = globals().get("db_engine", None)

    if not db_engine:
        # Host e Database
        host = "localhost"
        database = "filmes_em_cartaz"
        
        # Solicita usuário e senha de forma segura
        user = input("Digite seu usuário MySQL: ")
        password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
        
        # Cria a engine SQLAlchemy
        db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

    # Criando diretórios, se não existir
    os.makedirs("dados/metacritic/notas", exist_ok=True)
    os.makedirs("dados/metacritic/reviews", exist_ok=True)

    # -------------------------------------------------------------
    # 1. Busca todos os filmes já processados
    # -------------------------------------------------------------
    query = "SELECT nome_pt FROM notas_filmes_em_cartaz WHERE fonte = 'METACRITIC'"
    with db_engine.connect() as conn:
        nomes_processados = pd.read_sql(query, conn)['nome_pt'].tolist()

    # Filtra apenas os filmes que ainda não foram processados
    df_filmes_a_processar = df_filmes[~df_filmes['nome_pt'].isin(nomes_processados)].copy()
    total = len(df_filmes_a_processar)

    # Filmes não processados
    lista_erros = []

    if total == 0:
        print("Todos os filmes já foram processados!")
        return

    # -------------------------------------------------------------
    # 2. Loop sobre os filmes restantes
    # -------------------------------------------------------------
    for idx, row in enumerate(df_filmes_a_processar.itertuples(), 1):
        nome_filme_en = row.nome_en
        nome_filme_pt = row.nome_pt
        nome_filme_original = row.nome_original

        # Print de progresso
        print("-" * 60)
        print(f"[{idx}/{total}] Processando: '{nome_filme_en}'")

        # Normaliza o nome para arquivos
        nome_normalizado = re.sub(r"[^\w\d]+", "_", nome_filme_pt)

        # Busca reviews e nota
        df_reviews, nota_geral = buscar_reviews_metacritic(nome_filme_en)

        if df_reviews.empty:
            print(f"Nenhum review encontrado para '{nome_filme_en}'")
            lista_erros.append(nome_filme_en)
            continue

        # Adiciona coluna fonte e nome
        df_reviews['nome_en'] = nome_filme_en
        df_reviews['nome_pt'] = nome_filme_pt
        df_reviews['nome_original'] = nome_filme_original
        df_reviews['fonte'] = 'METACRITIC'

        # -------------------------------------------------------------
        # Salva reviews no SQL
        # -------------------------------------------------------------
        df_reviews_sql = df_reviews[['nome_pt', 'nome_original', 'nome_en', 'nota', 'review', 'fonte']]
        df_reviews_sql.to_sql(
            'reviews_filmes_em_cartaz',
            con=db_engine,
            if_exists='append',
            index=False,
            method="multi",
            chunksize=5000
        )

        # -------------------------------------------------------------
        # Salva nota no SQL e parquet
        # -------------------------------------------------------------
        if nota_geral is not None:
            df_nota_sql = pd.DataFrame({
                'nome_pt': [nome_filme_pt],
                'nome_original': [nome_filme_original],
                'nome_en': [nome_filme_en],
                'nota': [nota_geral],
                'fonte': ['METACRITIC']
            })

            df_nota_sql.to_sql(
                'notas_filmes_em_cartaz',
                con=db_engine,
                if_exists='append',
                index=False,
                method="multi",
                chunksize=5000
            )

            # Salvando arquivos .parquet
            df_nota_sql.to_parquet(f"dados/metacritic/notas/{nome_normalizado}.parquet", index=False)
            df_reviews_sql.to_parquet(f"dados/metacritic/reviews/{nome_normalizado}.parquet", index=False)

        print(f"Concluído: '{nome_filme_en}'\n")

        # Pausa entre requisições
        time.sleep(10)

    print("-" * 60)
    print("-" * 60)
    print("Processo Finalizado com sucesso!!!")
    print(f"{len(lista_erros)} Filmes sem Reviews")
    print("-" * 60)
    print("-" * 60)

In [24]:
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "filmes_em_cartaz"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

# Lista de filmes
query = "SELECT * FROM filmes_em_cartaz"
df_filmes = pd.read_sql(query, con=db_engine)
df_filmes_top_10 = df_filmes.sort_values(by = 'popularidade', ascending = False).head(10).copy()

In [25]:
df_filmes_top_10.shape

(10, 12)

In [26]:
df_filmes_top_10

,id,nome_pt,nome_original,nome_en,data_estreia,url_poster,caminho_imagem,sinopse,nota_media,num_avaliacoes,popularidade,created_at
0,1,Missão Refúgio,Shelter,Shelter,2026-03-12,https://image.tmdb.org/t/p/original/hSvhZRkbYD...,imagens/Missão Refúgio.jpeg,"Um ex-assassino de aluguel, agora um homem rec...",6.65,284,294.6049,2026-03-14 09:36:01
1,2,O Som da Morte,Whistle,Whistle,2026-02-05,https://image.tmdb.org/t/p/original/1tX11KQzuj...,imagens/O Som da Morte.jpeg,Um grupo de estudantes disfuncionais se depara...,6.13,131,141.9966,2026-03-14 09:36:01
2,3,"Cara de Um, Focinho de Outro",Hoppers,Hoppers,2026-03-05,https://image.tmdb.org/t/p/original/ftPFJBGbld...,"imagens/Cara de Um, Focinho de Outro.jpeg",Mabel é uma jovem amante da natureza que tenta...,7.70,158,139.8369,2026-03-14 09:36:01
3,4,Destruição Final 2,Greenland 2: Migration,Greenland 2: Migration,2026-02-05,https://image.tmdb.org/t/p/original/sgvNCaWe4L...,imagens/Destruição Final 2.jpeg,Tendo encontrado a segurança do bunker da Groe...,6.46,587,131.5286,2026-03-14 09:36:01
4,5,Pânico 7,Scream 7,Scream 7,2026-02-26,https://image.tmdb.org/t/p/original/oz4Mt34HH8...,imagens/Pânico 7.jpeg,Quando um novo Ghostface surge na pacata cidad...,5.95,297,91.6557,2026-03-14 09:36:01
5,6,O Morro dos Ventos Uivantes,Wuthering Heights,Wuthering Heights,2026-02-12,https://image.tmdb.org/t/p/original/762vOiAWzo...,imagens/O Morro dos Ventos Uivantes.jpeg,O órfão Heathcliff é acolhido ainda criança pe...,6.40,490,75.0383,2026-03-14 09:36:01
6,7,Caminhos do Crime,Crime 101,Crime 101,2026-02-12,https://image.tmdb.org/t/p/original/hHnEDdzdmK...,imagens/Caminhos do Crime.jpeg,Um ladrão enigmático realiza assaltos de alto ...,7.02,155,64.5261,2026-03-14 09:36:01
7,8,A Noiva!,The Bride!,The Bride!,2026-03-05,https://image.tmdb.org/t/p/original/vcdnSEcm0f...,imagens/A Noiva!.jpeg,"Na década de 1930, Frankenstein viaja a Chicag...",6.30,150,62.6113,2026-03-14 09:36:01
8,9,Um Cabra Bom de Bola,GOAT,GOAT,2026-02-12,https://image.tmdb.org/t/p/original/x0SRTrltSJ...,imagens/Um Cabra Bom de Bola.jpeg,Uma pequena cabra com grandes sonhos recebe um...,7.50,103,47.0737,2026-03-14 09:36:01
9,10,O Testamento de Ann Lee,The Testament of Ann Lee,The Testament of Ann Lee,2026-03-12,https://image.tmdb.org/t/p/original/mDiS3wF7sz...,imagens/O Testamento de Ann Lee.jpeg,"Ann Lee, a líder fundadora do movimento Shaker...",7.00,37,35.6559,2026-03-14 09:36:01


In [27]:
%%time
# Buscar e salvar notas e reviews
salvar_reviews_e_nota_metacritic(df_filmes_top_10)

------------------------------------------------------------
[1/10] Processando: 'Shelter'
Concluído: 'Shelter'

------------------------------------------------------------
[2/10] Processando: 'Whistle'
Concluído: 'Whistle'

------------------------------------------------------------
[3/10] Processando: 'Hoppers'
Concluído: 'Hoppers'

------------------------------------------------------------
[4/10] Processando: 'Greenland 2: Migration'
Concluído: 'Greenland 2: Migration'

------------------------------------------------------------
[5/10] Processando: 'Scream 7'
Concluído: 'Scream 7'

------------------------------------------------------------
[6/10] Processando: 'Wuthering Heights'
Concluído: 'Wuthering Heights'

------------------------------------------------------------
[7/10] Processando: 'Crime 101'
Concluído: 'Crime 101'

------------------------------------------------------------
[8/10] Processando: 'The Bride!'
Concluído: 'The Bride!'

----------------------------------

## Letterboxd

In [28]:
def buscar_reviews_letterboxd(nome_filme, max_reviews=50):
    """
    Descrição:
        Busca reviews de usuários de um filme no Letterboxd, incluindo a nota média geral.
        Retorna até `max_reviews` reviews, padronizando as notas na escala 1–10.
        Apenas reviews com nota e texto válidos são incluídos, e emojis são removidos.

    Parâmetros:
        nome_filme (str): Nome do filme a ser pesquisado no Letterboxd.
        max_reviews (int): Número máximo de reviews a serem retornadas.

    Processo:
        1. Abre o Letterboxd usando undetected-chromedriver.
        2. Pesquisa o filme pelo nome e seleciona o primeiro resultado.
        3. Extrai a nota média geral exibida na página e converte para a escala 1–10.
        4. Navega até a página de reviews do filme.
        5. Coleta os reviews e as notas dos usuários:
            - Remove emojis do texto.
            - Converte notas de 0–5 estrelas para a escala 1–10.
            - Descartar reviews sem nota ou texto válido.
        6. Percorre as páginas de reviews até atingir `max_reviews`.

    Retorno:
        tuple:
            - pandas.DataFrame: DataFrame com colunas ["nota", "review"], notas na escala 1–10.
            - float | None: Nota média geral do filme (escala 1–10).
    """

    # =============================================================
    # Função auxiliar para remover emojis
    # =============================================================
    def remove_emojis(text):
        # Emojis principais
        text = regex.sub(r'\p{Extended_Pictographic}', '', text)
        
        # Skin tones
        text = regex.sub(r'[\U0001F3FB-\U0001F3FF]', '', text)
        
        # Bandeiras
        text = regex.sub(r'[\U0001F1E6-\U0001F1FF]{2}', '', text)
        
        # Keycaps 
        text = regex.sub(r'[0-9#*]\uFE0F?\u20E3', '', text)
        
        # Caracteres invisíveis
        text = regex.sub(r'[\u200D\uFE0E\uFE0F]', '', text)
        
        # Limpeza final
        text = regex.sub(r'\s+', ' ', text).strip()
        
        return text

    # =============================================================
    # Função auxiliar para converter estrelas → escala 0-10
    # =============================================================
    def converter_estrelas(label):

        if not label:
            return None

        estrelas = label.count("★")
        meia = 0.5 if "½" in label else 0

        return (estrelas + meia) * 2

    # =============================================================
    # Abrir navegador (undetected chromedriver)
    # =============================================================
    options = Options()
    
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    
    driver = uc.Chrome(options=options,
                       version_main=145)

    driver.minimize_window()

    wait = WebDriverWait(driver, 30)

    try:

        # =============================================================
        # 1. Buscar filme
        # =============================================================
        query = urllib.parse.quote(nome_filme)

        url = f"https://letterboxd.com/search/{query}"

        driver.get(url)

        primeiro_resultado = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "h2.headline-2 a[href^='/film/']")
            )
        )

        link = primeiro_resultado.get_attribute("href")

        driver.get(link)

        time.sleep(2)

        # =============================================================
        # 2. Nota média
        # =============================================================
        try:

            nota = driver.find_element(
                By.CSS_SELECTOR,
                "span.average-rating"
            ).text

            nota_geral = float(nota) * 2

        except:
            nota_geral = None

        # =============================================================
        # 3. Abrir reviews
        # =============================================================
        driver.get(link + "reviews/")

        time.sleep(2)

        reviews = []
        ratings = []

        # =============================================================
        # 4. Coletar reviews
        # =============================================================
        while len(reviews) < max_reviews:

            itens = driver.find_elements(By.CSS_SELECTOR, "div.listitem")

            if not itens:
                break

            for item in itens:
            
                # ================== Review ==================
                try:
                    review = item.find_element(
                        By.CSS_SELECTOR,
                        "div.js-review-body"
                    ).text.strip()
                    review = remove_emojis(review)  # remove emojis
                    if not review:  # descarta vazios ou só emojis
                        review = None
                except:
                    review = None
            
                # ================== Rating ==================
                try:
                    label = item.find_element(
                        By.CSS_SELECTOR,
                        "span.inline-rating"
                    ).text
                    rating = converter_estrelas(label)
                    if rating is None:  # descarta notas inválidas
                        rating = None
                except:
                    rating = None
            
                # ================== Guardar ==================
                if review is not None and rating is not None:
                    reviews.append(review)
                    ratings.append(rating)

                if len(reviews) >= max_reviews:
                    break

            try:

                botao = driver.find_element(By.CSS_SELECTOR, "a.next")

                driver.execute_script(
                    "arguments[0].click();",
                    botao
                )

                time.sleep(3)

            except:
                break

        df_reviews = pd.DataFrame({
            "nota": ratings,
            "review": reviews
        })

    finally:
        driver.quit()

    return df_reviews, nota_geral

In [29]:
%%time
df_letterboxd, nota_letterboxd = buscar_reviews_letterboxd("Scream 7")

print("Nota Letterboxd:", nota_letterboxd)
df_letterboxd.head()

Nota Letterboxd: 4.6
CPU times: total: 734 ms
Wall time: 39.4 s


,nota,review
0,4.0,(AMC Braintree 10)
1,6.0,Au prochain épisode on va avoir droit au livre...
2,1.0,i only watched this bc i always watch the new ...
3,5.0,Casser le film impeu
4,5.0,c’était vrm drôle comme film mais j’ai quand m...


In [30]:
df_letterboxd.shape

(50, 2)

In [31]:
def salvar_reviews_e_nota_letterboxd(df_filmes):
    """
    Processa filmes do DataFrame fornecido, buscando notas e reviews no Letterboxd.
    
    Para cada filme na coluna 'nome' do DataFrame:
    1. Verifica se já existe uma nota do Letterboxd na tabela 'notas_filmes_em_cartaz'.
    2. Caso não exista, busca reviews e nota geral usando `buscar_reviews_letterboxd`.
    3. Salva os reviews na tabela 'reviews_filmes_em_cartaz' com a fonte 'LETTERBOXD'.
    4. Salva a nota geral na tabela 'notas_filmes_em_cartaz' com a fonte 'LETTERBOXD'.
    5. Gera arquivos parquet separados em 'dados/letterboxd/' para nota e reviews.
    
    Parâmetros:
        df_filmes (pd.DataFrame): DataFrame contendo pelo menos a coluna 'nome_pt'.
        db_engine (sqlalchemy.engine.Engine): Conexão com o banco de dados.
    
    Retorno:
        None
    """

    # Verificando conexão com o Banco de Dados    
    db_engine = globals().get("db_engine", None)

    if not db_engine:
        # Host e Database
        host = "localhost"
        database = "filmes_em_cartaz"
        
        # Solicita usuário e senha de forma segura
        user = input("Digite seu usuário MySQL: ")
        password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
        
        # Cria a engine SQLAlchemy
        db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

    # Criando diretórios, se não existir
    os.makedirs("dados/letterboxd/notas", exist_ok=True)
    os.makedirs("dados/letterboxd/reviews", exist_ok=True)

    # -------------------------------------------------------------
    # 1. Busca todos os filmes já processados
    # -------------------------------------------------------------
    query = "SELECT nome_pt FROM notas_filmes_em_cartaz WHERE fonte = 'LETTERBOXD'"
    with db_engine.connect() as conn:
        nomes_processados = pd.read_sql(query, conn)['nome_pt'].tolist()

    # Filtra apenas os filmes que ainda não foram processados
    df_filmes_a_processar = df_filmes[~df_filmes['nome_pt'].isin(nomes_processados)].copy()
    total = len(df_filmes_a_processar)

    # Filmes não processados
    lista_erros = []

    if total == 0:
        print("Todos os filmes já foram processados!")
        return

    # -------------------------------------------------------------
    # 2. Loop sobre os filmes restantes
    # -------------------------------------------------------------
    for idx, row in enumerate(df_filmes_a_processar.itertuples(), 1):
        nome_filme_en = row.nome_en
        nome_filme_pt = row.nome_pt
        nome_filme_original = row.nome_original

        # Print de progresso
        print("-" * 60)
        print(f"[{idx}/{total}] Processando: '{nome_filme_en}'")

        # Normaliza o nome para arquivos
        nome_normalizado = re.sub(r"[^\w\d]+", "_", nome_filme_pt)

        # Busca reviews e nota
        df_reviews, nota_geral = buscar_reviews_letterboxd(nome_filme_en)

        if df_reviews.empty:
            print(f"Nenhum review encontrado para '{nome_filme_en}'")
            lista_erros.append(nome_filme_en)
            continue

        # Adiciona coluna fonte e nome
        df_reviews['nome_en'] = nome_filme_en
        df_reviews['nome_pt'] = nome_filme_pt
        df_reviews['nome_original'] = nome_filme_original
        df_reviews['fonte'] = 'LETTERBOXD'

        # -------------------------------------------------------------
        # Salva reviews no SQL
        # -------------------------------------------------------------
        df_reviews_sql = df_reviews[['nome_pt', 'nome_original', 'nome_en', 'nota', 'review', 'fonte']]
        df_reviews_sql.to_sql(
            'reviews_filmes_em_cartaz',
            con=db_engine,
            if_exists='append',
            index=False,
            method="multi",
            chunksize=5000
        )

        # -------------------------------------------------------------
        # Salva nota no SQL e parquet
        # -------------------------------------------------------------
        if nota_geral is not None:
            df_nota_sql = pd.DataFrame({
                'nome_pt': [nome_filme_pt],
                'nome_original': [nome_filme_original],
                'nome_en': [nome_filme_en],
                'nota': [nota_geral],
                'fonte': ['LETTERBOXD']
            })

            df_nota_sql.to_sql(
                'notas_filmes_em_cartaz',
                con=db_engine,
                if_exists='append',
                index=False,
                method="multi",
                chunksize=5000
            )

            # Salvando arquivos .parquet
            df_nota_sql.to_parquet(f"dados/letterboxd/notas/{nome_normalizado}.parquet", index=False)
            df_reviews_sql.to_parquet(f"dados/letterboxd/reviews/{nome_normalizado}.parquet", index=False)

        print(f"Concluído: '{nome_filme_en}'\n")

        # Pausa entre requisições
        time.sleep(10)

    print("-" * 60)
    print("-" * 60)
    print("Processo Finalizado com sucesso!!!")
    print(f"{len(lista_erros)} Filmes sem Reviews")
    print("-" * 60)
    print("-" * 60)

In [32]:
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "filmes_em_cartaz"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

# Lista de filmes
query = "SELECT * FROM filmes_em_cartaz"
df_filmes = pd.read_sql(query, con=db_engine)
df_filmes_top_10 = df_filmes.sort_values(by = 'popularidade', ascending = False).head(10).copy()

In [33]:
df_filmes_top_10.shape

(10, 12)

In [34]:
df_filmes_top_10

,id,nome_pt,nome_original,nome_en,data_estreia,url_poster,caminho_imagem,sinopse,nota_media,num_avaliacoes,popularidade,created_at
0,1,Missão Refúgio,Shelter,Shelter,2026-03-12,https://image.tmdb.org/t/p/original/hSvhZRkbYD...,imagens/Missão Refúgio.jpeg,"Um ex-assassino de aluguel, agora um homem rec...",6.65,284,294.6049,2026-03-14 09:36:01
1,2,O Som da Morte,Whistle,Whistle,2026-02-05,https://image.tmdb.org/t/p/original/1tX11KQzuj...,imagens/O Som da Morte.jpeg,Um grupo de estudantes disfuncionais se depara...,6.13,131,141.9966,2026-03-14 09:36:01
2,3,"Cara de Um, Focinho de Outro",Hoppers,Hoppers,2026-03-05,https://image.tmdb.org/t/p/original/ftPFJBGbld...,"imagens/Cara de Um, Focinho de Outro.jpeg",Mabel é uma jovem amante da natureza que tenta...,7.70,158,139.8369,2026-03-14 09:36:01
3,4,Destruição Final 2,Greenland 2: Migration,Greenland 2: Migration,2026-02-05,https://image.tmdb.org/t/p/original/sgvNCaWe4L...,imagens/Destruição Final 2.jpeg,Tendo encontrado a segurança do bunker da Groe...,6.46,587,131.5286,2026-03-14 09:36:01
4,5,Pânico 7,Scream 7,Scream 7,2026-02-26,https://image.tmdb.org/t/p/original/oz4Mt34HH8...,imagens/Pânico 7.jpeg,Quando um novo Ghostface surge na pacata cidad...,5.95,297,91.6557,2026-03-14 09:36:01
5,6,O Morro dos Ventos Uivantes,Wuthering Heights,Wuthering Heights,2026-02-12,https://image.tmdb.org/t/p/original/762vOiAWzo...,imagens/O Morro dos Ventos Uivantes.jpeg,O órfão Heathcliff é acolhido ainda criança pe...,6.40,490,75.0383,2026-03-14 09:36:01
6,7,Caminhos do Crime,Crime 101,Crime 101,2026-02-12,https://image.tmdb.org/t/p/original/hHnEDdzdmK...,imagens/Caminhos do Crime.jpeg,Um ladrão enigmático realiza assaltos de alto ...,7.02,155,64.5261,2026-03-14 09:36:01
7,8,A Noiva!,The Bride!,The Bride!,2026-03-05,https://image.tmdb.org/t/p/original/vcdnSEcm0f...,imagens/A Noiva!.jpeg,"Na década de 1930, Frankenstein viaja a Chicag...",6.30,150,62.6113,2026-03-14 09:36:01
8,9,Um Cabra Bom de Bola,GOAT,GOAT,2026-02-12,https://image.tmdb.org/t/p/original/x0SRTrltSJ...,imagens/Um Cabra Bom de Bola.jpeg,Uma pequena cabra com grandes sonhos recebe um...,7.50,103,47.0737,2026-03-14 09:36:01
9,10,O Testamento de Ann Lee,The Testament of Ann Lee,The Testament of Ann Lee,2026-03-12,https://image.tmdb.org/t/p/original/mDiS3wF7sz...,imagens/O Testamento de Ann Lee.jpeg,"Ann Lee, a líder fundadora do movimento Shaker...",7.00,37,35.6559,2026-03-14 09:36:01


In [35]:
%%time
# Buscar e salvar notas e reviews
salvar_reviews_e_nota_letterboxd(df_filmes_top_10)

------------------------------------------------------------
[1/10] Processando: 'Shelter'
Concluído: 'Shelter'

------------------------------------------------------------
[2/10] Processando: 'Whistle'
Concluído: 'Whistle'

------------------------------------------------------------
[3/10] Processando: 'Hoppers'
Concluído: 'Hoppers'

------------------------------------------------------------
[4/10] Processando: 'Greenland 2: Migration'
Concluído: 'Greenland 2: Migration'

------------------------------------------------------------
[5/10] Processando: 'Scream 7'
Concluído: 'Scream 7'

------------------------------------------------------------
[6/10] Processando: 'Wuthering Heights'
Concluído: 'Wuthering Heights'

------------------------------------------------------------
[7/10] Processando: 'Crime 101'
Concluído: 'Crime 101'

------------------------------------------------------------
[8/10] Processando: 'The Bride!'
Concluído: 'The Bride!'

----------------------------------

# Reviews Traduzidos (Inglês e Português)

In [40]:
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "filmes_em_cartaz"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

# Lista de filmes
query = "SELECT * FROM reviews_filmes_em_cartaz"
df_reviews = pd.read_sql(query, con=db_engine)

In [41]:
df_reviews

,id,nome_pt,nome_original,nome_en,nota,review,fonte
0,1,Missão Refúgio,Shelter,Shelter,7.0,Shelter - Throwback Statham Action,IMDB
1,2,Missão Refúgio,Shelter,Shelter,6.0,"Shelter isn't bad, it's just familiar. A servi...",IMDB
2,3,Missão Refúgio,Shelter,Shelter,6.0,Disposable fun with a game Statham,IMDB
3,4,Missão Refúgio,Shelter,Shelter,8.0,A Solid Statham Flick.,IMDB
4,5,Missão Refúgio,Shelter,Shelter,7.0,Surprisingly good,IMDB
...,...,...,...,...,...,...,...
1710,1711,Caminhos do Crime,Crime 101,Crime 101,7.0,who knew thor could be a good thief,LETTERBOXD
1711,1712,Caminhos do Crime,Crime 101,Crime 101,8.0,This movie was a pleasant surprise. Some unnec...,LETTERBOXD
1712,1713,Caminhos do Crime,Crime 101,Crime 101,7.0,"A thing about me is, even if you make a movie ...",LETTERBOXD
1713,1714,Caminhos do Crime,Crime 101,Crime 101,5.0,barry keoghan will not return in avengers doom...,LETTERBOXD


In [70]:
def traduzir_reviews(df, coluna_texto="review", delay=0.2):
    """
    Descrição:
        Traduz automaticamente textos de reviews para português e inglês,
        além de identificar o idioma original de cada texto.
    
    Parâmetros:
        df (pandas.DataFrame): DataFrame contendo a coluna de reviews.
        coluna_texto (str): Nome da coluna com os textos a serem traduzidos.
        delay (float): Tempo de espera (em segundos) entre requisições para evitar bloqueios.
    
    Processo:
        1. Itera sobre cada texto da coluna especificada no DataFrame.
        2. Ignora valores nulos ou textos vazios.
        3. Detecta automaticamente o idioma original utilizando `langdetect`.
        4. Traduz o texto para inglês e português utilizando `GoogleTranslator`.
        5. Em caso de erro durante a detecção ou tradução, retorna valores nulos.
        6. Aplica um pequeno delay entre requisições para reduzir risco de bloqueio.
        7. Armazena os resultados em listas separadas.
    
    Retorno:
        pandas.DataFrame:
            DataFrame original acrescido das colunas:
                - review_en: Texto traduzido para inglês
                - review_pt: Texto traduzido para português
                - lingua_original: Idioma detectado do texto original
    """
    
    reviews_pt = []
    reviews_en = []
    linguas = []

    for texto in tqdm(df[coluna_texto], desc="Traduzindo reviews"):

        if pd.isna(texto) or not str(texto).strip():
            reviews_pt.append(None)
            reviews_en.append(None)
            linguas.append(None)
            continue

        texto = str(texto)

        try:
            try:
                lingua = detect(texto)
            except Exception:
                lingua = "unknown"

            traducao_en = GoogleTranslator(source='auto', target='en').translate(texto)
            traducao_pt = GoogleTranslator(source='auto', target='pt').translate(texto)

        except Exception:
            lingua = None
            traducao_en = None
            traducao_pt = None

        reviews_en.append(traducao_en)
        reviews_pt.append(traducao_pt)
        linguas.append(lingua)

        time.sleep(delay)

    df = df.copy()
    df["review_en"] = reviews_en
    df["review_pt"] = reviews_pt
    df["lingua_original"] = linguas

    return df

In [74]:
%%time
df_reviews_traduzidos = traduzir_reviews(df_reviews)

Traduzindo reviews: 100%|██████████████████████████████████████████████████████████| 1715/1715 [53:55<00:00,  1.89s/it]

CPU times: total: 15min 44s
Wall time: 53min 55s


In [75]:
df_reviews_traduzidos

,id,nome_pt,nome_original,nome_en,nota,review,fonte,review_en,review_pt,lingua_original
0,1,Missão Refúgio,Shelter,Shelter,7.0,Shelter - Throwback Statham Action,IMDB,Shelter - Throwback Statham Action,Abrigo - Ação Statham de retrocesso,en
1,2,Missão Refúgio,Shelter,Shelter,6.0,"Shelter isn't bad, it's just familiar. A servi...",IMDB,"Shelter isn't bad, it's just familiar. A servi...","Abrigo não é ruim, é apenas familiar. Um thril...",en
2,3,Missão Refúgio,Shelter,Shelter,6.0,Disposable fun with a game Statham,IMDB,Disposable fun with a game Statham,Diversão descartável com um jogo Statham,en
3,4,Missão Refúgio,Shelter,Shelter,8.0,A Solid Statham Flick.,IMDB,A Solid Statham Flick.,Um filme sólido de Statham.,en
4,5,Missão Refúgio,Shelter,Shelter,7.0,Surprisingly good,IMDB,Surprisingly good,Surpreendentemente bom,en
...,...,...,...,...,...,...,...,...,...,...
1710,1711,Caminhos do Crime,Crime 101,Crime 101,7.0,who knew thor could be a good thief,LETTERBOXD,who knew thor could be a good thief,quem diria que Thor poderia ser um bom ladrão,en
1711,1712,Caminhos do Crime,Crime 101,Crime 101,8.0,This movie was a pleasant surprise. Some unnec...,LETTERBOXD,This movie was a pleasant surprise. Some unnec...,Este filme foi uma grata surpresa. Algumas tra...,en
1712,1713,Caminhos do Crime,Crime 101,Crime 101,7.0,"A thing about me is, even if you make a movie ...",LETTERBOXD,"A thing about me is, even if you make a movie ...","Uma coisa sobre mim é que, mesmo que você faça...",en
1713,1714,Caminhos do Crime,Crime 101,Crime 101,5.0,barry keoghan will not return in avengers doom...,LETTERBOXD,barry keoghan will not return in avengers doom...,Barry Keoghan não retornará em Vingadores: O J...,en


## Inserir no Banco de Dados e Salvar Parquet

In [76]:
# Salvar Parquet
os.makedirs("processados/reviews", exist_ok=True)
df_reviews_traduzidos.to_parquet(f"processados/reviews/reviews_traduzidos.parquet", index=False)

# Inserir no banco de dados
df_reviews_traduzidos.to_sql(
    'reviews_traduzidos_filmes_em_cartaz',
    con=db_engine,
    if_exists='append',
    index=False,
    method="multi",
    chunksize=5000
)

1715

# Criar Wordcloud com os posteres do filmes

In [7]:
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "filmes_em_cartaz"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

# Lista de filmes
query = """
SELECT 
    r.*,
    f.caminho_imagem_pt,
    f.caminho_imagem_en
FROM reviews_traduzidos_filmes_em_cartaz r
LEFT JOIN filmes_em_cartaz f
    ON LOWER(r.nome_pt) = LOWER(f.nome_pt)
"""

df_reviews_traduzidos_full = pd.read_sql(query, con=db_engine)

In [8]:
df_reviews_traduzidos_full.shape

(1715, 12)

In [9]:
df_reviews_traduzidos_full

,id,nome_pt,nome_original,nome_en,nota,review,review_pt,review_en,lingua_original,fonte,caminho_imagem_pt,caminho_imagem_en
0,1,Missão Refúgio,Shelter,Shelter,7.0,Shelter - Throwback Statham Action,Abrigo - Ação Statham de retrocesso,Shelter - Throwback Statham Action,en,IMDB,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
1,2,Missão Refúgio,Shelter,Shelter,6.0,"Shelter isn't bad, it's just familiar. A servi...","Abrigo não é ruim, é apenas familiar. Um thril...","Shelter isn't bad, it's just familiar. A servi...",en,IMDB,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
2,3,Missão Refúgio,Shelter,Shelter,6.0,Disposable fun with a game Statham,Diversão descartável com um jogo Statham,Disposable fun with a game Statham,en,IMDB,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
3,4,Missão Refúgio,Shelter,Shelter,8.0,A Solid Statham Flick.,Um filme sólido de Statham.,A Solid Statham Flick.,en,IMDB,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
4,5,Missão Refúgio,Shelter,Shelter,7.0,Surprisingly good,Surpreendentemente bom,Surprisingly good,en,IMDB,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
...,...,...,...,...,...,...,...,...,...,...,...,...
1710,1711,Caminhos do Crime,Crime 101,Crime 101,7.0,who knew thor could be a good thief,quem diria que Thor poderia ser um bom ladrão,who knew thor could be a good thief,en,LETTERBOXD,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg
1711,1712,Caminhos do Crime,Crime 101,Crime 101,8.0,This movie was a pleasant surprise. Some unnec...,Este filme foi uma grata surpresa. Algumas tra...,This movie was a pleasant surprise. Some unnec...,en,LETTERBOXD,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg
1712,1713,Caminhos do Crime,Crime 101,Crime 101,7.0,"A thing about me is, even if you make a movie ...","Uma coisa sobre mim é que, mesmo que você faça...","A thing about me is, even if you make a movie ...",en,LETTERBOXD,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg
1713,1714,Caminhos do Crime,Crime 101,Crime 101,5.0,barry keoghan will not return in avengers doom...,Barry Keoghan não retornará em Vingadores: O J...,barry keoghan will not return in avengers doom...,en,LETTERBOXD,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg


In [12]:
nltk.download("stopwords")
nltk.download('averaged_perceptron_tagger_eng')

# =========================================================
# Normalização
# =========================================================
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    import unicodedata
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-z\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def normalizar_palavra(p):
    import unicodedata
    p = unicodedata.normalize("NFD", p)
    return p.encode("ascii", "ignore").decode("utf-8")

# =========================================================
# Stopwords (NLTK + WORDCLOUD + CUSTOMIZAÇÃO)
# =========================================================
from nltk.corpus import stopwords

def get_stopwords(idioma="pt"):
    stopwords_final = set(STOPWORDS)
    if idioma == "pt":
        stopwords_nltk = set(stopwords.words("portuguese"))
        stopwords_custom = {
            'filme','filmes','historia','personagem','personagens',
            'cena','cenas','diretor','atores','atriz','assistir','assistido',
            'de','a','o','que','e','do','da','em','um','para','é','com','não',
            'uma','os','no','se','na','por','mais','as','dos','como','mas','foi',
            'ao','ele','das','tem','à','seu','sua','ou','ser','quando','muito',
            'há','nos','já','está','eu','também','só','pela','até'
        }
        stopwords_final.update(stopwords_nltk)
        stopwords_final.update(stopwords_custom)
    elif idioma == "en":
        stopwords_nltk = set(stopwords.words("english"))
        stopwords_custom = {
            "movie","film","one","like","really","even",
            "would","watch","watched","story","character","characters"
        }
        stopwords_final.update(stopwords_nltk)
        stopwords_final.update(stopwords_custom)
    # normalizar stopwords
    stopwords_final = {normalizar_palavra(p) for p in stopwords_final}
    return stopwords_final

# =======================================================================
# Função auxiliar para gerar WordCloud de um grupo de reviews de um filme
# =======================================================================

# Carregar modelo spaCy para português
nlp_pt = spacy.load("pt_core_news_sm")

def gerar_wordcloud_filme(grupo, coluna_texto, coluna_caminho, pasta_saida, filme, fonte,
                          max_words, alpha_poster, contorno_offset, cor_contorno, stopwords):
    textos = grupo[coluna_texto].dropna().astype(str)
    if textos.empty:
        return

    # Normalizar
    textos = textos.apply(normalizar_texto)
    todas_palavras = " ".join(textos).split()

    # Detectar idioma do review (pega primeiro texto não vazio)
    try:
        idioma = detect(textos.iloc[0])
    except:
        idioma = "pt"  # fallback

    # Filtrar só adjetivos
    if idioma.startswith("en"):
        adjs = [word for (word, pos) in nltk.pos_tag(todas_palavras) if pos.startswith("JJ")]
    elif idioma.startswith("pt"):
        doc = nlp_pt(" ".join(todas_palavras))
        adjs = [token.text for token in doc if token.pos_ == "ADJ"]
    else:
        adjs = todas_palavras  # fallback, pega tudo

    if not adjs:
        return  # nada para gerar

    texto_unico = " ".join(adjs)

    # Caminho do poster
    caminho_poster = grupo[coluna_caminho].dropna().unique()
    if len(caminho_poster) == 0:
        print(f"Sem caminho de imagem: {filme}")
        return
    caminho_poster = caminho_poster[0]
    if not os.path.exists(caminho_poster):
        print(f"Imagem não encontrada: {caminho_poster}")
        return

    try:
        poster = Image.open(caminho_poster).convert("RGBA")
    except Exception as e:
        print(f"Erro ao abrir imagem: {caminho_poster} | {e}")
        return

    # Poster semi-transparente
    poster_np = np.array(poster)
    poster_np[..., 3] = alpha_poster
    poster_transparente = Image.fromarray(poster_np)

    # WordCloud principal
    wc = WordCloud(
        width=poster.width,
        height=poster.height,
        background_color=None,
        mode="RGBA",
        stopwords=stopwords,
        max_words=max_words,
        collocations=False
    ).generate(texto_unico)

    # Contorno
    wc_contorno_arr = np.array(wc.to_image().convert("RGBA"))
    wc_contorno_arr = np.roll(wc_contorno_arr, shift=contorno_offset, axis=0)
    wc_contorno_arr = np.roll(wc_contorno_arr, shift=contorno_offset, axis=1)
    mask = wc_contorno_arr[..., 3] > 0
    wc_contorno_arr[mask, :3] = cor_contorno
    wc_contorno_img = Image.fromarray(wc_contorno_arr)

    # WordCloud colorida
    image_colors = ImageColorGenerator(np.array(poster))
    wc_img = wc.recolor(color_func=image_colors).to_image().convert("RGBA")

    # Combinar
    resultado = Image.alpha_composite(poster_transparente, wc_contorno_img)
    resultado = Image.alpha_composite(resultado, wc_img)

    # Salvar
    nome_arquivo = f"{filme}_{fonte}.png".replace(" ", "_")
    nome_arquivo = re.sub(r'[<>:"/\\|?*\'"“”‘’]', '', nome_arquivo)
    caminho_saida = os.path.join(pasta_saida, nome_arquivo)
    resultado.save(caminho_saida)

# =========================================================
# Função principal
# =========================================================
def gerar_wordclouds_por_fonte(
    df,
    coluna_texto,
    coluna_fonte="fonte",
    coluna_filme="nome_pt",
    coluna_caminho="caminho_imagem_pt",
    pasta_saida="processados/poster_wordcloud",
    idioma="pt",
    max_words=50,
    alpha_poster=150,
    contorno_offset=2,
    cor_contorno=(0,0,0)
):
    pasta_saida = os.path.join(pasta_saida, idioma)
    os.makedirs(pasta_saida, exist_ok=True)
    stopwords = get_stopwords(idioma)

    # WordClouds por fonte
    grupos = df.groupby([coluna_filme, coluna_fonte])
    total = len(grupos)
    for i, ((filme, fonte), grupo) in enumerate(grupos, start=1):
        print(f"[{i}/{total}] {filme} - {fonte}")
        gerar_wordcloud_filme(grupo, coluna_texto, coluna_caminho, pasta_saida, filme, fonte,
                               max_words, alpha_poster, contorno_offset, cor_contorno, stopwords)

    # WordCloud GERAL (todas as fontes juntas)
    filmes_unicos = df[coluna_filme].dropna().unique()
    for i, filme in enumerate(filmes_unicos, start=1):
        grupo_filme = df[df[coluna_filme] == filme]
        print(f"[GERAL {i}/{len(filmes_unicos)}] {filme} - GERAL")
        gerar_wordcloud_filme(grupo_filme, coluna_texto, coluna_caminho, pasta_saida, filme, "GERAL",
                               max_words, alpha_poster, contorno_offset, cor_contorno, stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mht-1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mht-1\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


In [13]:
%%time
# Gera Posters em Português
gerar_wordclouds_por_fonte(
    df_reviews_traduzidos_full,
    coluna_texto="review_pt",
    coluna_filme="nome_pt",
    coluna_caminho="caminho_imagem_pt",
    idioma="pt",
    alpha_poster=50
)

[1/40] A Noiva! - IMDB
[2/40] A Noiva! - LETTERBOXD
[3/40] A Noiva! - METACRITIC
[4/40] A Noiva! - ROTTEN_TOMATOES
[5/40] Caminhos do Crime - IMDB
[6/40] Caminhos do Crime - LETTERBOXD
[7/40] Caminhos do Crime - METACRITIC
[8/40] Caminhos do Crime - ROTTEN_TOMATOES
[9/40] Cara de Um, Focinho de Outro - IMDB
[10/40] Cara de Um, Focinho de Outro - LETTERBOXD
[11/40] Cara de Um, Focinho de Outro - METACRITIC
[12/40] Cara de Um, Focinho de Outro - ROTTEN_TOMATOES
[13/40] Destruição Final 2 - IMDB
[14/40] Destruição Final 2 - LETTERBOXD
[15/40] Destruição Final 2 - METACRITIC
[16/40] Destruição Final 2 - ROTTEN_TOMATOES
[17/40] Missão Refúgio - IMDB
[18/40] Missão Refúgio - LETTERBOXD
[19/40] Missão Refúgio - METACRITIC
[20/40] Missão Refúgio - ROTTEN_TOMATOES
[21/40] O Morro dos Ventos Uivantes - IMDB
[22/40] O Morro dos Ventos Uivantes - LETTERBOXD
[23/40] O Morro dos Ventos Uivantes - METACRITIC
[24/40] O Morro dos Ventos Uivantes - ROTTEN_TOMATOES
[25/40] O Som da Morte - IMDB
[26/40] O

In [14]:
%%time
# Gera Posters em Inglês
gerar_wordclouds_por_fonte(
    df_reviews_traduzidos_full,
    coluna_texto="review_en",
    coluna_filme="nome_en",
    coluna_caminho="caminho_imagem_en",
    idioma="en",
    alpha_poster=50
)

[1/40] Crime 101 - IMDB
[2/40] Crime 101 - LETTERBOXD
[3/40] Crime 101 - METACRITIC
[4/40] Crime 101 - ROTTEN_TOMATOES
[5/40] GOAT - IMDB
[6/40] GOAT - LETTERBOXD
[7/40] GOAT - METACRITIC
[8/40] GOAT - ROTTEN_TOMATOES
[9/40] Greenland 2: Migration - IMDB
[10/40] Greenland 2: Migration - LETTERBOXD
[11/40] Greenland 2: Migration - METACRITIC
[12/40] Greenland 2: Migration - ROTTEN_TOMATOES
[13/40] Hoppers - IMDB
[14/40] Hoppers - LETTERBOXD
[15/40] Hoppers - METACRITIC
[16/40] Hoppers - ROTTEN_TOMATOES
[17/40] Scream 7 - IMDB
[18/40] Scream 7 - LETTERBOXD
[19/40] Scream 7 - METACRITIC
[20/40] Scream 7 - ROTTEN_TOMATOES
[21/40] Shelter - IMDB
[22/40] Shelter - LETTERBOXD
[23/40] Shelter - METACRITIC
[24/40] Shelter - ROTTEN_TOMATOES
[25/40] The Bride! - IMDB
[26/40] The Bride! - LETTERBOXD
[27/40] The Bride! - METACRITIC
[28/40] The Bride! - ROTTEN_TOMATOES
[29/40] The Testament of Ann Lee - IMDB
[30/40] The Testament of Ann Lee - LETTERBOXD
[31/40] The Testament of Ann Lee - METACRITIC


# Gerando HTML consolidado

In [3]:
# =============================
# Geração completa dos HTMLs dos filmes + index
# =============================
def gerar_html_completo(df, pasta_filmes="filmes_html"):
    """
    Descrição:
        Gera páginas HTML individuais para cada filme presente no DataFrame
        e cria um arquivo index.html com links para todos os filmes, com suporte a PT/EN.

    Parâmetros:
        df (pandas.DataFrame): DataFrame contendo os dados dos filmes.
        pasta_filmes (str): Nome da pasta onde os HTMLs individuais serão salvos.

    Processo:
        1. Cria a pasta de saída caso não exista.
        2. Obtém a lista de filmes únicos com base na coluna 'nome_pt'.
        3. Itera sobre cada filme:
            - Filtra os dados do filme.
            - Obtém os caminhos dos posters em português e inglês.
            - Valida a existência do poster em português.
            - Gera o HTML do filme utilizando `gerar_html_filme`.
            - Cria um nome de arquivo seguro.
            - Salva o HTML na pasta especificada.
            - Define os dados em inglês (com fallback).
            - Armazena os dados necessários para o index.
        4. Gera o arquivo index.html utilizando `gerar_index`.

    Retorno:
        None:
            Os arquivos HTML são salvos no diretório especificado.
    """

    # =============================
    # Criação da pasta de saída
    # =============================
    os.makedirs(pasta_filmes, exist_ok=True)

    # =============================
    # Lista de filmes únicos
    # =============================
    filmes = df["nome_pt"].dropna().unique()
    links = []

    print("Gerando html's dos filmes...")

    # =============================
    # Loop principal por filme
    # =============================
    for filme in filmes:
        df_filme = df[df["nome_pt"] == filme]

        # =============================
        # Caminhos dos posters (PT e EN)
        # =============================
        caminho_poster_pt = df_filme["caminho_imagem_pt"].dropna().unique()

        if "caminho_imagem_en" in df_filme.columns:
            caminho_poster_en = df_filme["caminho_imagem_en"].dropna().unique()
        else:
            caminho_poster_en = caminho_poster_pt

        # =============================
        # Validação de imagem
        # =============================
        if len(caminho_poster_pt) == 0:
            print(f"Sem imagem: {filme}")
            continue

        # =============================
        # Geração do HTML do filme
        # =============================
        html_filme = gerar_html_filme(
            df_filme,
            caminho_poster_pt[0],
            caminho_poster_en[0]
        )

        # =============================
        # Tratamento do nome do arquivo
        # =============================
        nome_arquivo = f"{filme.replace(' ', '_')}.html"
        nome_arquivo = re.sub(r'[<>:"/\\|?*\'"“”‘’]', '', nome_arquivo)
        caminho_html = f"{pasta_filmes}/{nome_arquivo}"

        # =============================
        # Salvando o HTML
        # =============================
        with open(caminho_html, "w", encoding="utf-8") as f:
            f.write(html_filme)

        # =============================
        # Dados em inglês (fallback)
        # =============================
        if "nome_en" in df_filme.columns:
            filme_en = df_filme["nome_en"].dropna().iloc[0]
        else:
            filme_en = filme

        link_en = caminho_html
        poster_en = caminho_poster_en[0]

        # =============================
        # Armazena dados para index
        # =============================
        links.append((
            filme,
            caminho_poster_pt[0],
            caminho_html,
            filme_en,
            poster_en,
            link_en
        ))

    # =============================
    # Geração do index.html
    # =============================
    print("Gerando index.html...")

    html_index = gerar_index(links)

    with open("index.html", "w", encoding="utf-8") as f:
        f.write(html_index)

In [4]:
# =============================
# Geração do index HTML do portfólio de filmes
# =============================
def gerar_index(links):
    """
    Descrição:
        Gera o arquivo HTML principal (index.html) do portfólio de filmes,
        exibindo os cards dos filmes com suporte a troca de idioma (PT/EN).

    Parâmetros:
        links (list): Lista de tuplas contendo os dados dos filmes no formato:
            (filme_pt, poster_pt, link_pt, filme_en, poster_en, link_en)

    Processo:
        1. Inicializa uma string para armazenar os cards HTML.
        2. Itera sobre cada tupla da lista de links:
            - Cria um card HTML com atributos data-* para PT e EN.
            - Define imagem, título e link do filme.
        3. Insere os cards dentro do layout HTML principal.
        4. Adiciona estilos CSS para layout responsivo.
        5. Implementa lógica em JavaScript para:
            - Detectar idioma via parâmetro da URL.
            - Alternar entre português e inglês.
            - Atualizar dinamicamente posters e títulos.
            - Controlar navegação ao clicar no filme.
        6. Retorna o HTML completo como string.

    Retorno:
        str:
            String contendo o HTML completo do index do portfólio.
    """
    cards_html = ""
    for filme_pt, poster_pt, link_pt, filme_en, poster_en, link_en in links:
        cards_html += f"""
        <div class="card"
             data-nome-pt="{filme_pt}" data-poster-pt="{poster_pt}" data-link-pt="{link_pt}"
             data-nome-en="{filme_en}" data-poster-en="{poster_en}" data-link-en="{link_en}">
            <img src="{poster_pt}" alt="{filme_pt}" onclick="abrirFilme(this)">
            <p class="titulo">{filme_pt}</p>
        </div>
        """

    return f"""
<html>
<head>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
body {{ font-family: Arial; background:#111; color:white; margin:0; padding:16px; }}
h1 {{ text-align:center; padding:20px; }}
.botao-grupo {{ margin:10px 0; display:flex; justify-content:center; gap:10px; }}
.btn-idioma {{ padding:8px 12px; background:#222; color:white; border:1px solid #555; border-radius:6px; cursor:pointer; }}
.btn-idioma:hover {{ background:#444; }}
.btn-selecionado {{ background:gold; color:black; border:1px solid white; }}
.grid {{ display:grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap:16px; padding:16px; }}
.card {{ text-align:center; }}
.card img {{ width:100%; height:auto; border-radius:10px; transition: transform 0.2s; cursor:pointer; }}
.card img:hover {{ transform: scale(1.05); }}
p.titulo {{ font-size:14px; }}
</style>
</head>
<body>
<h1 id="titulo">Filmes</h1>

<div class="botao-grupo">
    <button class="btn-idioma" id="btn-pt" onclick="setLang('pt')">PT</button>
    <button class="btn-idioma" id="btn-en" onclick="setLang('en')">EN</button>
</div>

<div class="grid">
    {cards_html}
</div>

<script>
let langAtual = "pt";

// Detecta parâmetro lang da URL
const urlParams = new URLSearchParams(window.location.search);
const langParam = urlParams.get('lang');
if(langParam === 'en') langAtual = 'en';

// Atualiza botões de idioma
function atualizarBotoes() {{
    document.querySelectorAll('.btn-idioma').forEach(btn => {{
        btn.classList.remove('btn-selecionado');
        if(btn.id === "btn-" + langAtual) btn.classList.add('btn-selecionado');
    }});
    // Atualiza grid com posters e títulos
    document.querySelectorAll('.card').forEach(card => {{
        const img = card.querySelector('img');
        const p = card.querySelector('.titulo');
        if(langAtual === 'pt') {{
            img.src = card.dataset.posterPt;
            p.innerText = card.dataset.nomePt;
        }} else {{
            img.src = card.dataset.posterEn;
            p.innerText = card.dataset.nomeEn;
        }}
    }});
    document.getElementById('titulo').innerText = langAtual === 'pt' ? 'Filmes' : 'Movies';
}}

// Ao clicar no poster
function abrirFilme(img) {{
    const card = img.parentElement;
    const link = langAtual === 'pt' ? card.dataset.linkPt : card.dataset.linkEn;
    window.location.href = link + "?lang=" + langAtual;
}}

function setLang(l) {{
    langAtual = l;
    atualizarBotoes();
}}

document.addEventListener("DOMContentLoaded", function() {{
    atualizarBotoes();
}});
</script>
</body>
</html>
"""

In [5]:
# =============================
# Geração do HTML individual do filme
# =============================
def gerar_html_filme(df_filme, caminho_poster_pt, caminho_poster_en, lang="pt"):
    """
    Descrição:
        Gera a página HTML completa de um filme específico, incluindo poster,
        nota média, botões interativos, tabela de reviews e wordcloud dinâmica,
        com suporte a múltiplas fontes e idiomas (PT/EN).

    Parâmetros:
        df_filme (pandas.DataFrame): DataFrame contendo os dados do filme.
        caminho_poster_pt (str): Caminho da imagem do poster em português.
        caminho_poster_en (str): Caminho da imagem do poster em inglês.
        lang (str): Idioma inicial da página ("pt" ou "en").

    Processo:
        1. Define as fontes disponíveis (GERAL, IMDb, Rotten Tomatoes, etc.).
        2. Calcula a nota média do filme com base na coluna 'nota'.
        3. Gera os botões de seleção de fontes dinamicamente.
        4. Gera os botões de seleção de idioma (PT/EN).
        5. Cria a tabela inicial utilizando a função `gerar_tabela_html`.
        6. Normaliza o nome do filme para uso em JavaScript e paths.
        7. Constrói o HTML completo incluindo:
            - Layout responsivo com CSS.
            - Poster do filme.
            - Barra de nota dinâmica.
            - Wordcloud inicial.
            - Tabela de reviews.
        8. Implementa funções JavaScript para:
            - Alternar entre fontes e idiomas.
            - Atualizar nota, tabela, poster e wordcloud dinamicamente.
            - Controlar navegação de volta ao menu.
        9. Injeta os dados processados via `gerar_dados_js` no frontend.

    Retorno:
        str:
            String contendo o HTML completo da página do filme.
    """

    fontes = ["GERAL", "IMDb", "Rotten Tomatoes", "Metacritic", "Letterboxd"]

    # Nota geral
    df_notas = df_filme.dropna(subset=["nota"])
    nota_geral = round(df_notas["nota"].mean(), 2) if not df_notas.empty else 0

    # Botões de fontes
    botoes_fontes = ""
    for f in fontes:
        botoes_fontes += f'<button class="btn-fonte" id="btn-{f}" data-fonte="{f}" onclick="setFonte(\'{f}\')">{f}</button>'

    # Botões de idioma
    botoes_idioma = '<button class="btn-idioma" id="btn-pt" onclick="setLang(\'pt\')">PT</button>' + \
                     '<button class="btn-idioma" id="btn-en" onclick="setLang(\'en\')">EN</button>'

    # Tabela inicial
    tabela_inicial = gerar_tabela_html(df_filme, "GERAL", lang)

    # Nome do filme para JS (substituindo espaços e caracteres problemáticos)
    filme_js_pt = df_filme['nome_pt'].iloc[0].replace(' ', '_').replace('/', '_')
    filme_js_en = df_filme['nome_en'].iloc[0].replace(' ', '_').replace('/', '_')
    filme_js_pt = re.sub(r'[<>:"/\\|?*\'"“”‘’]', '', filme_js_pt)
    filme_js_en = re.sub(r'[<>:"/\\|?*\'"“”‘’]', '', filme_js_en)

    html = f"""<html>
<head>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
body {{ font-family: Arial; background:#111; color:white; margin:0; padding:16px; }}
.container {{ max-width:1000px; margin:auto; text-align:center; }}
#voltar {{ text-align:left; margin-bottom:20px; }}
#voltar button {{ padding:10px 16px; font-size:16px; }}
h1 {{ text-align:center; margin:10px 0; }}
.botao-grupo {{ margin: 10px 0; display:flex; justify-content:center; flex-wrap:wrap; gap:10px; }}
.btn-fonte, .btn-idioma {{ padding:8px 12px; background:#222; color:white; border:1px solid #555; border-radius:6px; cursor:pointer; }}
.btn-fonte:hover, .btn-idioma:hover {{ background:#444; }}
.btn-selecionado {{ background:gold; color:black; border:1px solid white; }}
.poster img {{ width:300px; max-width:80%; border-radius:10px; margin:20px 0; }}
.wordcloud img {{ width:100%; max-width:600px; height:auto; margin:20px auto; display:block; border-radius:10px; }}
.table-container {{ overflow-x:auto; margin-top:20px; }}
table {{ width:100%; border-collapse:collapse; }}
th, td {{ padding:8px; border-bottom:1px solid #444; font-size:14px; }}
.nota {{ width:100%; max-width:300px; background:#333; border-radius:10px; overflow:hidden; height:30px; margin:10px auto; position:relative; }}
.nota-bar {{ height:100%; width:0%; border-radius:10px; font-weight:bold; font-size:16px; color:black; text-align:center; line-height:30px; transition: width 0.5s ease; }}
</style>
</head>
<body>
<div class="container">
    <div id="voltar">
        <button onclick="voltarMenu()">🔙 Menu</button>
    </div>
    <h1 id="titulo_filme">{df_filme['nome_pt'].iloc[0]}</h1>

    <div class="botao-grupo" id="botoes_fontes">{botoes_fontes}</div>
    <div class="botao-grupo" id="botoes_idioma">{botoes_idioma}</div>

    <div class="poster">
        <img id="poster" src="../{caminho_poster_pt}">
    </div>

    <div id="nota_barra" class="nota">
        <div class="nota-bar"></div>
    </div>

    <div id="conteudo" class="conteudo">
        <div class="wordcloud">
            <img src="../processados/poster_wordcloud/{filme_js_pt}_GERAL.png" alt="WordCloud">
        </div>
        <div class="table-container">{tabela_inicial}</div>
    </div>
</div>

<script>
// Variáveis globais JS
let fonteAtual = "GERAL";
let langAtual = "{lang}";  

function _corNota(valor) {{
    if(valor <= 4) return "red";
    else if(valor <= 6) return "orange";
    else if(valor <= 8) return "yellow";
    else return "green";
}}

const urlParams = new URLSearchParams(window.location.search);
const langParam = urlParams.get('lang');
if(langParam === 'en') langAtual = 'en';

function setFonte(f) {{ fonteAtual = f; atualizar(); atualizarBotoes(); }}
function setLang(l) {{ langAtual = l; atualizar(); atualizarBotoes(); }}

function voltarMenu() {{ window.location.href = "../index.html?lang=" + langAtual; }}

function atualizar() {{
    let key = fonteAtual + "_" + langAtual;
    let data = window.dados[key];
    let fonteFormatada = fonteAtual
        .toUpperCase()
        .replace(/\s+/g, "_");
    if(!data) return;

    document.getElementById("titulo_filme").innerText = data.nome;

    const notaBar = document.getElementById("nota_barra").querySelector(".nota-bar");
    const cor = _corNota(data.nota);
    notaBar.style.width = (data.nota*10) + "%";
    notaBar.style.background = cor;
    notaBar.innerText = data.nota;

    let nomeFilmeWC = (langAtual === "pt") ? window.filme_pt : window.filme_en;
    let wcNome = encodeURIComponent(nomeFilmeWC) + "_" + fonteFormatada + ".png";
    let wcPath = `../processados/poster_wordcloud/${{langAtual}}/${{wcNome}}`;
    document.getElementById("conteudo").innerHTML = `<div class="wordcloud"><img src="${{wcPath}}" alt="WordCloud"></div><div class="table-container">${{data.tabela}}</div>`;

    document.getElementById("poster").src = "../" + data.poster;
}}

function atualizarTextoBotoes() {{
    document.querySelectorAll('.btn-fonte').forEach(btn => {{
        if(btn.dataset.fonte === "GERAL") {{
            btn.innerText = langAtual === "pt" ? "Todos" : "All";
        }}
    }});
}}

function atualizarBotoes() {{
    document.querySelectorAll('.btn-fonte').forEach(btn => {{
        btn.classList.remove('btn-selecionado');
        if(btn.dataset.fonte === fonteAtual) btn.classList.add('btn-selecionado');
    }});
    document.querySelectorAll('.btn-idioma').forEach(btn => {{
        btn.classList.remove('btn-selecionado');
        if(btn.id === "btn-" + langAtual) btn.classList.add('btn-selecionado');
    }});
    atualizarTextoBotoes();
}}

document.addEventListener("DOMContentLoaded", function() {{
    atualizar();
    atualizarBotoes();
}});
</script>

<script>
window.dados = {gerar_dados_js(df_filme, caminho_poster_pt, caminho_poster_en)};
window.filme_pt = "{filme_js_pt}";
window.filme_en = "{filme_js_en}";
</script>
</body>
</html>
"""
    return html

In [6]:
# =============================
# Geração dos dados em JavaScript para o HTML do filme
# =============================
def gerar_dados_js(df_filme, poster_pt, poster_en):
    """
    Descrição:
        Gera um objeto em formato JSON contendo os dados do filme organizados
        por fonte e idioma, para uso dinâmico no frontend (JavaScript).

    Parâmetros:
        df_filme (pandas.DataFrame): DataFrame contendo os dados do filme.
        poster_pt (str): Caminho do poster em português.
        poster_en (str): Caminho do poster em inglês.

    Processo:
        1. Define as fontes disponíveis (GERAL, IMDb, Rotten Tomatoes, etc.).
        2. Define os idiomas suportados (pt e en).
        3. Itera sobre cada combinação de fonte e idioma:
            - Define o nome do filme conforme o idioma (com fallback).
            - Define o poster correspondente ao idioma (com fallback).
            - Filtra as notas conforme a fonte selecionada.
            - Calcula a nota média do filme.
            - Gera a tabela HTML utilizando `gerar_tabela_html`.
            - Armazena os dados em um dicionário estruturado.
        4. Converte o dicionário final para uma string JSON.

    Retorno:
        str:
            String JSON contendo os dados estruturados por fonte e idioma,
            no formato esperado pelo JavaScript do HTML.
    """
    js_obj = {}
    fontes = ["GERAL", "IMDb", "Rotten Tomatoes", "Metacritic", "Letterboxd"]
    idiomas = ["pt", "en"]

    # =============================
    # Loop por fonte e idioma
    # =============================
    for f in fontes:
        for lang in idiomas:

            # =============================
            # Definição de nome e poster
            # =============================
            if lang == "pt":
                nome = df_filme["nome_pt"].dropna().iloc[0]
                poster = poster_pt
            else:
                if "nome_en" in df_filme.columns and not df_filme["nome_en"].dropna().empty:
                    nome = df_filme["nome_en"].dropna().iloc[0]
                else:
                    nome = df_filme["nome_pt"].iloc[0]

                poster = poster_en if poster_en else poster_pt

            # =============================
            # Cálculo da nota média
            # =============================
            if f == "GERAL":
                df_notas = df_filme
            else:
                df_notas = df_filme[df_filme["fonte"] == f]

            df_notas = df_notas.dropna(subset=["nota"])
            nota = round(df_notas["nota"].mean(), 2) if not df_notas.empty else 0

            # =============================
            # Geração da tabela HTML
            # =============================
            tabela = gerar_tabela_html(df_filme, f, lang)

            # =============================
            # Armazena no objeto final
            # =============================
            js_obj[f"{f}_{lang}"] = {
                "nome": nome,
                "nota": nota,
                "tabela": tabela,
                "poster": poster
            }

    # =============================
    # Conversão para JSON
    # =============================
    return json.dumps(js_obj)

In [7]:
# =============================
# Geração da tabela HTML de reviews do filme
# =============================
def gerar_tabela_html(df_filme, fonte_selecionada="GERAL", lang="pt"):
    """
    Descrição:
        Gera uma tabela HTML contendo as notas e reviews de um filme,
        podendo filtrar por fonte específica e alternar entre idiomas (PT/EN).

    Parâmetros:
        df_filme (pandas.DataFrame): DataFrame contendo os dados do filme com colunas:
            ['fonte', 'nota', 'review_pt', 'review_en'].
        fonte_selecionada (str): Fonte específica para filtro ou "GERAL" para todas.
        lang (str): Idioma da tabela ("pt" ou "en").

    Processo:
        1. Define os nomes das colunas da tabela conforme o idioma selecionado.
        2. Filtra o DataFrame pela fonte escolhida (caso não seja "GERAL").
        3. Inicializa a estrutura HTML da tabela.
        4. Define o cabeçalho da tabela:
            - Inclui a coluna de fonte apenas se "GERAL" estiver selecionado.
        5. Itera sobre cada linha do DataFrame:
            - Adiciona os valores de fonte (se aplicável), nota e review.
            - Seleciona a coluna de review conforme o idioma.
        6. Finaliza a estrutura HTML da tabela.

    Retorno:
        str:
            String contendo o HTML completo da tabela de reviews.
    """

    # =============================
    # Definição dos nomes das colunas
    # =============================
    if lang == "pt":
        col_nota = "Nota"
        col_review = "Comentário"
        col_source = "Fonte"
    else:
        col_nota = "Rating"
        col_review = "Review"
        col_source = "Source"

    # =============================
    # Filtro por fonte
    # =============================
    if fonte_selecionada != "GERAL":
        df = df_filme[df_filme["fonte"] == fonte_selecionada].copy()
    else:
        df = df_filme.copy()

    # =============================
    # Inicialização da tabela
    # =============================
    html = '<table>'
    html += '<tr>'

    # =============================
    # Cabeçalho da tabela
    # =============================
    if fonte_selecionada == "GERAL":
        html += f'<th>{col_source}</th>'
    html += f'<th>{col_nota}</th>'
    html += f'<th>{col_review}</th>'
    html += '</tr>'

    # =============================
    # Preenchimento das linhas
    # =============================
    for _, row in df.iterrows():
        html += '<tr>'

        if fonte_selecionada == "GERAL":
            html += f'<td>{row["fonte"]}</td>'

        html += f'<td>{row["nota"]}</td>'

        review_col = "review_pt" if lang == "pt" else "review_en"
        review_text = row[review_col] if review_col in row else ""

        html += f'<td>{review_text}</td>'
        html += '</tr>'

    # =============================
    # Finalização da tabela
    # =============================
    html += '</table>'

    return html

In [8]:
db_engine = globals().get("db_engine", None)

# Configuração do MySQL 
if not db_engine:
    # Host e Database
    host = "localhost"
    database = "filmes_em_cartaz"
    
    # Solicita usuário e senha de forma segura
    user = input("Digite seu usuário MySQL: ")
    password = getpass.getpass("Digite sua senha MySQL (não será exibida): ")
    
    # Cria a engine SQLAlchemy
    db_engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{database}")

# Lista de filmes
query = """
SELECT 
    r.*,
    f.caminho_imagem_pt,
    f.caminho_imagem_en
FROM reviews_traduzidos_filmes_em_cartaz r
LEFT JOIN filmes_em_cartaz f
    ON LOWER(r.nome_pt) = LOWER(f.nome_pt)
"""

df_reviews_traduzidos_full = pd.read_sql(query, con=db_engine)

# Renomeando Fontes
df_reviews_traduzidos_full['fonte'] = df_reviews_traduzidos_full['fonte'].replace({"IMDB": "IMDb",
                                                                                   "ROTTEN_TOMATOES": "Rotten Tomatoes",
                                                                                   "METACRITIC": "Metacritic",
                                                                                   "LETTERBOXD": "Letterboxd"})

In [9]:
df_reviews_traduzidos_full

,id,nome_pt,nome_original,nome_en,nota,review,review_pt,review_en,lingua_original,fonte,caminho_imagem_pt,caminho_imagem_en
0,1,Missão Refúgio,Shelter,Shelter,7.0,Shelter - Throwback Statham Action,Abrigo - Ação Statham de retrocesso,Shelter - Throwback Statham Action,en,IMDb,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
1,2,Missão Refúgio,Shelter,Shelter,6.0,"Shelter isn't bad, it's just familiar. A servi...","Abrigo não é ruim, é apenas familiar. Um thril...","Shelter isn't bad, it's just familiar. A servi...",en,IMDb,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
2,3,Missão Refúgio,Shelter,Shelter,6.0,Disposable fun with a game Statham,Diversão descartável com um jogo Statham,Disposable fun with a game Statham,en,IMDb,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
3,4,Missão Refúgio,Shelter,Shelter,8.0,A Solid Statham Flick.,Um filme sólido de Statham.,A Solid Statham Flick.,en,IMDb,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
4,5,Missão Refúgio,Shelter,Shelter,7.0,Surprisingly good,Surpreendentemente bom,Surprisingly good,en,IMDb,imagens/pt/Missão Refúgio.jpeg,imagens/en/Shelter.jpeg
...,...,...,...,...,...,...,...,...,...,...,...,...
1710,1711,Caminhos do Crime,Crime 101,Crime 101,7.0,who knew thor could be a good thief,quem diria que Thor poderia ser um bom ladrão,who knew thor could be a good thief,en,Letterboxd,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg
1711,1712,Caminhos do Crime,Crime 101,Crime 101,8.0,This movie was a pleasant surprise. Some unnec...,Este filme foi uma grata surpresa. Algumas tra...,This movie was a pleasant surprise. Some unnec...,en,Letterboxd,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg
1712,1713,Caminhos do Crime,Crime 101,Crime 101,7.0,"A thing about me is, even if you make a movie ...","Uma coisa sobre mim é que, mesmo que você faça...","A thing about me is, even if you make a movie ...",en,Letterboxd,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg
1713,1714,Caminhos do Crime,Crime 101,Crime 101,5.0,barry keoghan will not return in avengers doom...,Barry Keoghan não retornará em Vingadores: O J...,barry keoghan will not return in avengers doom...,en,Letterboxd,imagens/pt/Caminhos do Crime.jpeg,imagens/en/Crime 101.jpeg


In [10]:
gerar_html_completo(df_reviews_traduzidos_full)

Gerando html's dos filmes...
Gerando index.html...
